<style>
.cl-responsive,
.cl-responsive * {
  box-sizing: border-box !important;
  max-width: 100% !important;
}
.cl-responsive {
  width: 100% !important;
  overflow-x: hidden !important;
  white-space: normal !important;
  overflow-wrap: anywhere !important;
  word-break: normal !important;
}
.cl-responsive code {
  white-space: normal !important;
  overflow-wrap: anywhere !important;
  word-break: break-word !important;
}
.cl-cover h1 {
  font-size: clamp(28px, 4vw, 42px) !important;
  line-height: 1.15 !important;
}
.cl-cover p {
  max-width: min(900px, 100%) !important;
}
</style>

<div class="cl-responsive cl-cover" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;padding:32px 36px;border-radius:16px;background:linear-gradient(135deg,#102A43 0%,#243B53 100%);color:#F7F4EC;margin:0 0 24px 0;">
  <div style="font-size:13px;letter-spacing:.16em;text-transform:uppercase;color:#D4A44C;font-weight:700;overflow-wrap:anywhere;">Notebook 02 · Panel Build</div>
  <h1 style="margin:10px 0 8px 0;font-size:clamp(28px,4vw,42px);line-height:1.15;font-weight:700;color:#FFFFFF;overflow-wrap:anywhere;word-break:normal;">ConflictLens — Country-Year Analytical Panel</h1>
  <p style="margin:0;max-width:min(900px,100%);font-size:18px;line-height:1.55;color:#D9E2EC;overflow-wrap:anywhere;word-break:normal;">Build a clean, auditable and reusable analysis-unit-year panel before any substantive country-year analysis.</p>
  <div style="margin-top:24px;padding-top:16px;border-top:1px solid rgba(255,255,255,.20);font-size:13px;color:#BCCCDC;overflow-wrap:anywhere;word-break:normal;">Python · Polars · pyarrow/Parquet · pycountry · Data Quality · Country-Year Panel</div>
</div>


## 0. Purpose, scope and reader guide

This notebook is a **panel-construction notebook**. It is not yet the country-year analysis notebook.

Its goal is to turn heterogeneous conflict and context sources into a single analytical backbone where each row represents one `analysis_unit_id + year`. The notebook therefore focuses on **keys, country harmonisation, temporal coverage, source joins, missing-value semantics, exports and documentation**.

The intended reader should be able to follow two layers at the same time:

| Reading layer | What the reader should understand |
|---|---|
| Methodological layer | Why the panel is built this way, what assumptions are avoided, and what remains unresolved. |
| Technical layer | How each source is loaded, harmonised, aggregated, joined and exported. |

A recurring design principle is that the code should never silently hide a methodological decision. When a choice is not safe to force — for example ambiguous historical countries or contested territories — the notebook exports a diagnostic rather than pretending the issue is solved.


<a id="executive-summary"></a>

## Executive summary

Notebook 02 implements **Phase 2A-ter — final validation/correction gate** for the ConflictLens analysis-unit-year panel build. It keeps the original Phase 2A objective — one auditable panel row per `analysis_unit_id + year` — while adding the final safeguards needed before Notebook 03.

The notebook produces two complementary views of the same panel logic:

| Output layer | Role |
|---|---|
| `staging` | Full, auditable panel. It keeps source-specific details, raw labels, diagnostic flags, source-universe flags and intermediate indicators. |
| `analysis_ready` | Curated panel for the next descriptive country-year notebook. It keeps the variables needed for the first analytical pass plus the flags required to interpret missingness and zero-filled conflict metrics. |

The most important methodological choices are:

- the panel is built from a **complete country-year backbone**, not from conflict rows only;
- **UCDP** is the primary conflict source;
- **ACLED** is treated as a complementary benchmark;
- the analytical time window is selected **after** coverage diagnostics;
- `0` and `NA` remain semantically distinct;
- selected conflict metrics are zero-filled only when the **source covers the year and the country belongs to the harmonised source universe**;
- historical and composite entities that cannot be mapped without interpretive assumptions are **documented as outside the analysis-unit panel with ISO3 as an optional attribute**, not silently assigned to successor states;
- optional enrichment is explicitly deferred to Phase 2BIS.

The initial `outputs/02_country_crosswalk_unmatched.csv` issue is no longer an open blocker in this V1 build. It is retained as a residual quality-control output and should be empty after the validated arbitration table is applied.


## Navigation

1. [Executive summary](#executive-summary)
2. [Methodological contract](#project-framing)
3. [Environment](#environment)
4. [Technical helpers](#helper-functions)
5. [Source map and loading](#source-loading)
6. [Coverage diagnostics before selecting a window](#coverage-diagnostics)
7. [Country harmonisation and crosswalk audit](#country-crosswalk)
8. [Complete country-year backbone](#complete-backbone)
9. [Source-specific transformations](#source-transformations)
10. [Panel joins and diagnostics](#panel-joins)
11. [Staging, zero-filling and column-selection policy](#panel-outputs)
12. [Data dictionary and exports](#data-dictionary)
13. [Validation and handoff](#validation-handoff)

The notebook follows a repeated narrative rhythm: **problem → rule → code → interpretation → consequence for the next step**. Code cells are therefore surrounded by markdown so the reader can understand why each transformation exists and how to read its result.

<a id="project-framing"></a>

<div class="cl-responsive cl-section-block" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 1</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">Methodological contract</h2>
  <p style='margin:8px 0 0 0;color:#D9E2EC;line-height:1.45;'>The rules that define what this notebook is allowed to do.</p>
</div>


The panel is not only a data object; it is a methodological contract. Before loading any source, we make explicit what the notebook will and will not infer.

| Dimension | Decision | Consequence for the notebook |
|---|---|---|
| Unit of analysis | One row per `analysis_unit_id + year` | Every source must be transformed or joined at that grain. |
| Backbone | Complete country-year grid | Country-years with no observed conflict remain visible. |
| Main conflict source | UCDP | UCDP variables anchor the conflict measurement layer. |
| Benchmark source | ACLED | ACLED is used for comparison, not to redefine the primary conflict concept. |
| Temporal window | Diagnosed before selection | The notebook does not hard-code the analysis window upfront. |
| `0` versus `NA` | Kept semantically distinct | Zero-filling is conditional on both source temporal coverage and country source-universe membership. |
| Country matching | Conservative matching plus validated V1 arbitration | Ambiguous cases are either mapped, excluded, or documented outside the analysis-unit panel through an auditable arbitration table. |
| Output design | Staging + analysis-ready | Auditability and usability are separated. |
| Optional enrichment | Deferred to Phase 2BIS | The first panel remains small, controlled and explainable. |

The practical implication is that the notebook is allowed to resolve cases only when the rule is explicit, auditable and documented. Historically or politically ambiguous labels are not silently folded into contemporary successor states.


### Output contract

This notebook writes the required panel outputs and final validation artefacts.

| Output | Purpose |
|---|---|
| `outputs/02_country_year_panel_staging.parquet` / `.csv` | Full audit panel. |
| `outputs/02_country_year_panel_analysis_ready.parquet` / `.csv` | Curated panel for the next country-year analysis notebook. |
| `outputs/02_country_year_data_dictionary.csv` | Variable-level documentation. |
| `outputs/02_country_year_join_report.csv` | Source loading, transformation and join audit. |
| `outputs/02_country_crosswalk_arbitration_v1.csv` | Validated country-crosswalk arbitration table. |
| `outputs/02_country_crosswalk_unmatched.csv` | Residual unmatched labels after arbitration; expected to be empty in this V1 build. |
| `outputs/02_zero_fill_impact_report.csv` | Audit of zeros prevented by the stricter country-universe zero-fill rule. |
| `outputs/02_zero_fill_excluded_country_universe.csv` | Country/source combinations deliberately not zero-filled because the country is outside the harmonised source universe. |
| `outputs/02_crosswalk_arbitration_impact_report.csv` | Row/fatality impact summary of mapped, excluded and outside-ISO3 arbitration decisions. |
| `outputs/02_ucdp_os_multicountry_allocation_report.csv` | Diagnostic of UCDP One-sided multi-country allocation. |

The column-selection decision is documented by the notebook narrative and by the generated data dictionary.


<a id="environment"></a>

<div class="cl-responsive cl-section-block" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 2</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">Environment</h2>
  <p style='margin:8px 0 0 0;color:#D9E2EC;line-height:1.45;'>Resolve paths and verify the Polars-based execution stack.</p>
</div>


This notebook uses `polars` as the main tabular engine. Excel files are read through Polars’ Excel backend (`fastexcel`), and Parquet/CSV exports are written directly from Polars dataframes. The environment cell intentionally performs a hard dependency check so that failures happen before any transformation logic is executed.

The path resolution expects a repository-like structure with a `data/` folder and an `outputs/` folder at the project root. This makes the notebook rerunnable locally without relying on hidden state from the authoring environment.


### Why this cell exists — environment contract

Before we inspect or transform any data, we need to make the execution context explicit. A panel-build notebook should fail early if a required dependency is missing, if paths are unresolved, or if the output directory cannot be created.

This cell therefore answers a technical but important reproducibility question: **can this notebook be rerun from a clean project checkout with the declared Polars stack and the expected `data/` / `outputs/` structure?**

In [1]:
from __future__ import annotations

import importlib.metadata as importlib_metadata
import importlib.util
import json
import platform
import re
import sys
import unicodedata
import warnings
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile
from typing import Any

from IPython.display import HTML, Markdown, display

REQUIRED_MODULES = {
    "polars": "polars",
    "fastexcel": "fastexcel",
    "pycountry": "pycountry",
}

missing = [module for module in REQUIRED_MODULES if importlib.util.find_spec(module) is None]
if missing:
    packages = " ".join(REQUIRED_MODULES[module] for module in missing)
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n\n"
        "Install the project requirements from the repository root, or install the missing packages directly:\n"
        f"    python -m pip install {packages}"
    )

import polars as pl
import pycountry

warnings.filterwarnings("ignore")
pl.Config.set_tbl_cols(80)
pl.Config.set_tbl_rows(100)
pl.Config.set_fmt_str_lengths(120)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    for candidate in [*PROJECT_ROOT.parents]:
        if (candidate / "data").exists():
            PROJECT_ROOT = candidate
            break

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_DIR.exists():
    raise FileNotFoundError(
        "data/ not found. Run the notebook from the repository root, or place it in a folder under the repository root. "
        "Expected structure: repo_root/data/."
    )

package_versions = {module: importlib_metadata.version(package) for module, package in REQUIRED_MODULES.items()}

display(Markdown(
    "<div style='padding:14px 16px;border-left:4px solid #4C78A8;background:#DCEEFF;border-radius:6px;line-height:1.55;'>"
    "<strong>Execution environment check.</strong><br>"
    f"Python: <code>{sys.version.split()[0]}</code> · Platform: <code>{platform.platform()}</code><br>"
    f"Project root resolved to: <code>{PROJECT_ROOT}</code><br>"
    f"Data directory exists: <code>{DATA_DIR.exists()}</code> · Path: <code>{DATA_DIR}</code><br>"
    "Packages: " + ", ".join([f"<code>{k} {v}</code>" for k, v in package_versions.items()]) +
    "</div>"
))

<div style='padding:14px 16px;border-left:4px solid #4C78A8;background:#DCEEFF;border-radius:6px;line-height:1.55;'><strong>Execution environment check.</strong><br>Python: <code>3.12.10</code> · Platform: <code>Windows-11-10.0.26200-SP0</code><br>Project root resolved to: <code>.</code><br>Data directory exists: <code>True</code> · Path: <code>.\data</code><br>Packages: <code>polars 1.42.1</code>, <code>fastexcel 0.20.2</code>, <code>pycountry 26.2.16</code></div>

### What this output tells us

The environment note should confirm three things: the project root has been resolved, the `data/` directory exists, and the required Polars-based dependencies are installed. If this cell fails, the rest of the notebook should not be trusted because paths or dependencies are not reproducible.

No analytical data has been transformed yet; this is purely the execution contract.

<a id="helper-functions"></a>

<div class="cl-responsive cl-section-block" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 3</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">Technical helpers</h2>
  <p style='margin:8px 0 0 0;color:#D9E2EC;line-height:1.45;'>Implementation support used by later methodological steps.</p>
</div>


The next cells define reusable helpers for reading files, normalising country names, resolving ISO3 codes and displaying diagnostic notes.

These helpers are intentionally grouped early so that the rest of the notebook can focus on the panel-building logic. The important methodological detail is the conservative country matcher: it allows safe aliases but blocks ambiguous historical or contested labels until the crosswalk diagnostic is reviewed.


### Why this cell exists — readable diagnostics and shared text cleaning

The notebook will repeatedly show compact diagnostic notes and normalise country labels from several sources. Without shared helper functions, those operations would be repeated and the matching logic would become harder to audit.

This cell defines display and text-normalisation helpers only. It does not make any substantive data decision yet.

In [2]:
NOTE_CSS = """<style>
.cl-note,
.cl-note * {
    box-sizing: border-box !important;
    max-width: 100% !important;
}
.cl-note {
    box-sizing: border-box !important;
    width: 100% !important;
    max-width: 100% !important;
    overflow-x: hidden !important;
    white-space: normal !important;
    overflow-wrap: anywhere !important;
    word-break: normal !important;
}
.cl-note code {
    white-space: normal !important;
    overflow-wrap: anywhere !important;
    word-break: break-word !important;
}
</style>"""

NOTE_STYLE = (
    "box-sizing:border-box;"
    "width:100%;"
    "max-width:100%;"
    "overflow-x:hidden;"
    "padding:14px 16px;"
    "border-left:4px solid #4C78A8;"
    "background:#DCEEFF;"
    "border-radius:6px;"
    "line-height:1.55;"
    "white-space:normal;"
    "overflow-wrap:anywhere;"
    "word-break:normal;"
)


def display_note(text: str) -> None:
    display(HTML(f"{NOTE_CSS}<div class='cl-note' style='{NOTE_STYLE}'>{text}</div>"))


def normalize_country_name(value: Any) -> str | None:
    if value is None:
        return None
    try:
        if isinstance(value, float) and value != value:
            return None
    except Exception:
        pass
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none", "null"}:
        return None
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    text = text.replace("&", " and ")
    text = re.sub(r"[’`´]", "'", text)
    text = re.sub(r"[^A-Za-z0-9() ,.'-]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text or None


def zip_csv_names(path: Path) -> list[str]:
    with ZipFile(path) as archive:
        return [name for name in archive.namelist() if name.lower().endswith(".csv")]


def read_csv_from_zip(path: Path, columns: list[str] | None = None) -> pl.DataFrame:
    with ZipFile(path) as archive:
        inner = zip_csv_names(path)[0]
        data = archive.read(inner)
    try:
        return pl.read_csv(BytesIO(data), columns=columns, infer_schema_length=10000, ignore_errors=True)
    except Exception:
        return pl.read_csv(BytesIO(data), columns=columns, infer_schema_length=10000, ignore_errors=True, encoding="latin1")


def read_excel_table(path: Path, usecols: list[str] | None = None) -> pl.DataFrame:
    df = pl.read_excel(path)
    if usecols is not None:
        missing = [col for col in usecols if col not in df.columns]
        if missing:
            raise ValueError(f"Missing Excel columns in {path.name}: {missing}")
        df = df.select(usecols)
    return df


def min_max_df(df: pl.DataFrame, column: str) -> tuple[int | None, int | None]:
    row = df.select(
        pl.col(column).cast(pl.Int64, strict=False).min().alias("min"),
        pl.col(column).cast(pl.Int64, strict=False).max().alias("max"),
    ).row(0)
    return (None if row[0] is None else int(row[0]), None if row[1] is None else int(row[1]))

### How to interpret this cell

This helper cell normally has no table output. Its success means the notebook can display readable diagnostic notes and apply the same country-name normalisation logic everywhere.

The key methodological benefit is consistency: every source will be cleaned with the same normalisation function rather than ad hoc source-specific string rules.

### Why this cell exists — conservative country matching rules

Country harmonisation is not a neutral string-cleaning task. Some labels are safe aliases; others are historical, contested, or geographically ambiguous. The panel should not silently force those cases into an ISO3 code.

This cell defines the ISO3 universe, safe aliases, ambiguous labels and matching function used later by the crosswalk diagnostic. Its role is to encode the project rule: **safe matches are allowed, ambiguous matches remain unresolved until review.**

In [3]:
SPECIAL_ISO3 = {"XKX": "Kosovo"}
VALID_ISO3 = {country.alpha_3 for country in pycountry.countries} | set(SPECIAL_ISO3)

def iso3_to_name(iso3: str) -> str | None:
    if iso3 in SPECIAL_ISO3:
        return SPECIAL_ISO3[iso3]
    c = pycountry.countries.get(alpha_3=iso3)
    return c.name if c else None

def build_pycountry_lookup() -> dict[str, str]:
    lookup = {}
    for c in pycountry.countries:
        for attr in ["name", "official_name", "common_name"]:
            val = getattr(c, attr, None)
            if val:
                lookup[normalize_country_name(val)] = c.alpha_3
    return lookup

PYCOUNTRY_NAME_TO_ISO3 = build_pycountry_lookup()
SAFE_COUNTRY_ALIASES = {
    "bahamas": "BHS", "the bahamas": "BHS", "bolivia": "BOL", "bosnia-herzegovina": "BIH",
    "bosnia and herzegovina": "BIH", "brunei": "BRN", "burma": "MMR", "myanmar": "MMR",
    "myanmar (burma)": "MMR", "cabo verde": "CPV", "cape verde": "CPV", "cambodia (kampuchea)": "KHM",
    "caribbean netherlands": "BES", "congo": "COG", "republic of congo": "COG", "republic of the congo": "COG",
    "cote d'ivoire": "CIV", "cote divoire": "CIV", "dr congo": "COD", "dr congo (zaire)": "COD",
    "democratic republic of congo": "COD", "democratic republic of the congo": "COD", "drc": "COD",
    "east timor": "TLS", "timor-leste": "TLS", "eswatini": "SWZ", "kingdom of eswatini (swaziland)": "SWZ",
    "swaziland": "SWZ", "falkland islands": "FLK", "gambia": "GMB", "the gambia": "GMB", "guernsey": "GGY",
    "bailiwick of guernsey": "GGY", "jersey": "JEY", "bailiwick of jersey": "JEY", "iran": "IRN",
    "ivory coast": "CIV", "laos": "LAO", "macedonia": "MKD", "north macedonia": "MKD",
    "madagascar (malagasy)": "MDG", "micronesia": "FSM", "moldova": "MDA", "north korea": "PRK",
    "palestine": "PSE", "russia": "RUS", "saint-barthelemy": "BLM", "saint barthelemy": "BLM",
    "saint-martin": "MAF", "saint martin": "MAF", "samoa (western samoa)": "WSM", "sint maarten": "SXM",
    "south korea": "KOR", "syria": "SYR", "tanzania": "TZA", "turkey": "TUR", "turkiye": "TUR",
    "united kingdom": "GBR", "uk": "GBR", "u.k.": "GBR", "united states": "USA", "united states of america": "USA",
    "vatican city": "VAT", "vatican city state": "VAT", "venezuela": "VEN", "vietnam": "VNM",
    "vietnam (north vietnam)": "VNM", "zimbabwe (rhodesia)": "ZWE",
}
AMBIGUOUS_COUNTRY_NAMES = {
    "kosovo", "russia (soviet union)", "serbia (yugoslavia)", "somaliland",
    "yemen (north yemen)", "yemen (south yemen)", "north yemen", "south yemen",
}

def fuzzy_country_candidates(raw_country: str, limit: int = 5) -> str:
    try:
        results = pycountry.countries.search_fuzzy(str(raw_country))
    except LookupError:
        return ""
    return "; ".join(f"{r.alpha_3}:{r.name}" for r in results[:limit])

def match_country(raw_country: Any) -> dict[str, Any]:
    norm = normalize_country_name(raw_country)
    if norm is None:
        return {"raw_country_norm": None, "country_iso3": None, "country_name": None, "match_status": "missing_raw_country", "match_method": None, "candidate_matches": ""}
    if norm in AMBIGUOUS_COUNTRY_NAMES:
        return {"raw_country_norm": norm, "country_iso3": None, "country_name": None, "match_status": "ambiguous", "match_method": "blocked_ambiguous_name", "candidate_matches": fuzzy_country_candidates(str(raw_country))}
    if norm in PYCOUNTRY_NAME_TO_ISO3:
        iso3 = PYCOUNTRY_NAME_TO_ISO3[norm]
        return {"raw_country_norm": norm, "country_iso3": iso3, "country_name": iso3_to_name(iso3), "match_status": "matched", "match_method": "pycountry_exact", "candidate_matches": ""}
    if norm in SAFE_COUNTRY_ALIASES:
        iso3 = SAFE_COUNTRY_ALIASES[norm]
        return {"raw_country_norm": norm, "country_iso3": iso3, "country_name": iso3_to_name(iso3), "match_status": "matched", "match_method": "safe_alias", "candidate_matches": ""}
    return {"raw_country_norm": norm, "country_iso3": None, "country_name": None, "match_status": "unmatched", "match_method": "no_conservative_match", "candidate_matches": fuzzy_country_candidates(str(raw_country))}

### How to interpret this cell

This cell also has little or no visible output. Its importance is in the rule it encodes: safe country aliases are mapped, while ambiguous historical or contested labels are blocked from automatic ISO3 assignment.

That conservative behaviour is what later produces the unmatched crosswalk file instead of hiding country-matching assumptions inside the code.

<a id="source-loading"></a>

<div class="cl-responsive cl-section-block" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 4</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">Source map and loading</h2>
  <p style='margin:8px 0 0 0;color:#D9E2EC;line-height:1.45;'>Before transforming data, clarify what each source contributes.</p>
</div>


The panel combines conflict sources and contextual sources. They do not all have the same grain, role or limitations.

| Source | Initial grain | Role in Notebook 02 | Main limitation handled here |
|---|---|---|---|
| UCDP Organized Violence CY | Country-year | Primary conflict country-year source | Already aggregated, but country names still need harmonisation. |
| UCDP GED | Event-level | Event-derived diagnostic aggregated to country-year | Requires aggregation and country matching. |
| UCDP One-sided | Actor/location/year | Civilian-violence diagnostic | Multi-country locations require allocation logic. |
| ACLED summaries | Country-year | Complementary benchmark | Different definitions and partial 2026 coverage. |
| V-Dem | Country-year | Minimal political/institutional context | Several correlated indicators; only a small V1 subset is retained. |
| WDI | Country-year | Minimal socioeconomic context | Missingness varies strongly by country and indicator. |

The code below loads only the columns required for this notebook. Loading a small subset is not meant to erase information; it keeps Notebook 02 focused on the minimal Phase 2A panel contract.


### Why this cell exists — verify and load the raw sources

The methodological plan assumes that UCDP, ACLED, V-Dem and WDI sources are available locally. Before constructing the panel, we need to confirm that these files are actually present and readable.

This cell loads the raw inputs and creates a compact loading manifest. At this stage, the sources are **not comparable yet**: they still have different grains, country labels and temporal coverage.

In [4]:
# paths/load
UCDP_OV_PATH=DATA_DIR/'ucdp'/'organizedviolencecy-261-csv.zip'
UCDP_GED_PATH=DATA_DIR/'ucdp'/'ged261-csv.zip'
UCDP_OS_PATH=DATA_DIR/'ucdp'/'ucdp-onesided-261-csv.zip'
ACLED_EVENTS_PATH=DATA_DIR/'acled'/'number_of_political_violence_events_by_country-year_as-of-26Jun2026.xlsx'
ACLED_FATALITIES_PATH=DATA_DIR/'acled'/'number_of_reported_fatalities_by_country-year_as-of-26Jun2026.xlsx'
ACLED_CIV_EVENTS_PATH=DATA_DIR/'acled'/'number_of_events_targeting_civilians_by_country-year_as-of-26Jun2026.xlsx'
ACLED_CIV_FATALITIES_PATH=DATA_DIR/'acled'/'number_of_reported_civilian_fatalities_by_country-year_as-of-26Jun2026.xlsx'
VDEM_PATH=DATA_DIR/'V-Dem-CY-FullOthers-v16_csv'/'V-Dem-CY-Full+Others-v16.csv'
WDI_PATH=DATA_DIR/'wdi_downloader_package'/'data'/'processed'/'world_bank_wdi_panel_wide.csv'
MANUAL_CROSSWALK_PATH=DATA_DIR/'country_crosswalk_manual.csv'

ucdp_ov=read_csv_from_zip(UCDP_OV_PATH)
ucdp_ged=read_csv_from_zip(UCDP_GED_PATH, ['id','year','type_of_violence','conflict_new_id','dyad_new_id','country','country_id','region','deaths_a','deaths_b','deaths_civilians','deaths_unknown','best','high','low'])
ucdp_os=read_csv_from_zip(UCDP_OS_PATH, ['conflict_id','dyad_id','actor_id','actor_name','year','best_fatality_estimate','low_fatality_estimate','high_fatality_estimate','is_government_actor','location','region'])
acled_events=read_excel_table(ACLED_EVENTS_PATH, ['COUNTRY','YEAR','EVENTS'])
acled_fatalities=read_excel_table(ACLED_FATALITIES_PATH, ['COUNTRY','YEAR','FATALITIES'])
acled_civ_events=read_excel_table(ACLED_CIV_EVENTS_PATH, ['COUNTRY','YEAR','EVENTS'])
acled_civ_fatalities=read_excel_table(ACLED_CIV_FATALITIES_PATH, ['COUNTRY','YEAR','FATALITIES'])
vdem=pl.read_csv(VDEM_PATH, columns=['country_name','country_text_id','country_id','year','v2x_polyarchy','v2x_libdem','v2x_delibdem','v2x_egaldem','v2x_rule','v2x_corr'], infer_schema_length=10000, ignore_errors=True)
wdi=pl.read_csv(WDI_PATH, infer_schema_length=10000, ignore_errors=True)

loaded_shapes = pl.DataFrame([
    {"source": "UCDP Organized Violence CY", "rows": ucdp_ov.height, "columns": ucdp_ov.width},
    {"source": "UCDP GED events subset", "rows": ucdp_ged.height, "columns": ucdp_ged.width},
    {"source": "UCDP One-sided subset", "rows": ucdp_os.height, "columns": ucdp_os.width},
    {"source": "ACLED political violence events CY", "rows": acled_events.height, "columns": acled_events.width},
    {"source": "ACLED reported fatalities CY", "rows": acled_fatalities.height, "columns": acled_fatalities.width},
    {"source": "ACLED civilian targeting events CY", "rows": acled_civ_events.height, "columns": acled_civ_events.width},
    {"source": "ACLED civilian fatalities CY", "rows": acled_civ_fatalities.height, "columns": acled_civ_fatalities.width},
    {"source": "V-Dem selected indicators", "rows": vdem.height, "columns": vdem.width},
    {"source": "WDI selected package", "rows": wdi.height, "columns": wdi.width},
])

loaded_shapes

source,rows,columns
str,i64,i64
"""UCDP Organized Violence CY""",7132,74
"""UCDP GED events subset""",417968,15
"""UCDP One-sided subset""",1408,11
"""ACLED political violence events CY""",2754,3
"""ACLED reported fatalities CY""",2983,3
"""ACLED civilian targeting events CY""",2717,3
"""ACLED civilian fatalities CY""",2717,3
"""V-Dem selected indicators""",28092,10
"""WDI selected package""",9620,11


### What this output tells us

The loading manifest is the first evidence that the expected sources are available. A successful result should show one row per raw input and confirm that each source has non-zero rows.

This still does **not** mean the sources are analytically compatible. Compatibility is established later through coverage diagnostics, country harmonisation and duplicate-key checks.

**Reading the loading table.** This first diagnostic confirms that all sources were found and parsed. It does not yet say whether the sources are comparable; that requires coverage diagnostics, country harmonisation and join checks in the next sections.


<a id="coverage-diagnostics"></a>

<div class="cl-responsive cl-section-block" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 5</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">Coverage diagnostics before selecting a window</h2>
  <p style='margin:8px 0 0 0;color:#D9E2EC;line-height:1.45;'>The analytical window is a result of diagnostics, not a prior assumption.</p>
</div>


A common mistake in panel construction is to choose a time window too early. Here, the notebook first inspects observed coverage source by source.

This matters because different datasets have different coverage horizons. For example, a context source may cover many decades, while a conflict benchmark may include a current partial year. If we force a window before checking coverage, we risk either dropping usable data or comparing years that are not comparable.

The diagnostic below answers three questions:

1. Which years are observed in each source?
2. Which sources define the broad staging window?
3. Which sources define the recommended analysis-ready window?


### Why this cell exists — diagnose coverage before choosing years

The project decision is that the time window should not be fixed before coverage is diagnosed. This cell computes observed coverage for every source: rows, years, country-label counts and source role.

The output is used to separate two ideas: a broad staging window for auditability, and a stricter analysis-ready window for the first descriptive notebook.

In [5]:
def source_coverage_row(source, family, role, unit, df, year_col, country_col=None):
    y_min,y_max=min_max_df(df, year_col)
    raw_country_values = None
    if country_col and country_col in df.columns:
        raw_country_values = int(df.select(pl.col(country_col).cast(pl.Utf8).str.strip_chars().n_unique()).item())
    return {'source':source,'family':family,'role':role,'unit_of_analysis':unit,'rows':df.height,'year_min':y_min,'year_max':y_max,'n_years_observed':int(df.select(pl.col(year_col).cast(pl.Int64,strict=False).n_unique()).item()),'raw_country_values':raw_country_values}
coverage_rows=[
source_coverage_row('ucdp_organized_violence_cy','UCDP','primary_conflict_source','analysis-unit-year',ucdp_ov,'year','country'),
source_coverage_row('ucdp_ged_events','UCDP','event_derived_diagnostic','event',ucdp_ged,'year','country'),
source_coverage_row('ucdp_one_sided','UCDP','civilian_violence_diagnostic','actor-location-year',ucdp_os,'year','location'),
source_coverage_row('acled_political_violence_events','ACLED','benchmark','analysis-unit-year',acled_events,'YEAR','COUNTRY'),
source_coverage_row('acled_reported_fatalities','ACLED','benchmark','analysis-unit-year',acled_fatalities,'YEAR','COUNTRY'),
source_coverage_row('acled_civilian_targeting_events','ACLED','benchmark','analysis-unit-year',acled_civ_events,'YEAR','COUNTRY'),
source_coverage_row('acled_reported_civilian_fatalities','ACLED','benchmark','analysis-unit-year',acled_civ_fatalities,'YEAR','COUNTRY'),
source_coverage_row('vdem_selected_context','V-Dem','minimal_context','analysis-unit-year',vdem,'year','country_name'),
source_coverage_row('wdi_selected_context','WDI','minimal_context','analysis-unit-year',wdi,'year','country')]
source_coverage=pl.DataFrame(coverage_rows).sort(['family','source'])

source_coverage

source,family,role,unit_of_analysis,rows,year_min,year_max,n_years_observed,raw_country_values
str,str,str,str,i64,i64,i64,i64,i64
"""acled_civilian_targeting_events""","""ACLED""","""benchmark""","""analysis-unit-year""",2717,1997,2026,30,250
"""acled_political_violence_events""","""ACLED""","""benchmark""","""analysis-unit-year""",2754,1997,2026,30,250
"""acled_reported_civilian_fatalities""","""ACLED""","""benchmark""","""analysis-unit-year""",2717,1997,2026,30,250
"""acled_reported_fatalities""","""ACLED""","""benchmark""","""analysis-unit-year""",2983,1997,2026,30,250
"""ucdp_ged_events""","""UCDP""","""event_derived_diagnostic""","""event""",417968,1989,2025,37,126
"""ucdp_one_sided""","""UCDP""","""civilian_violence_diagnostic""","""actor-location-year""",1408,1989,2025,37,183
"""ucdp_organized_violence_cy""","""UCDP""","""primary_conflict_source""","""analysis-unit-year""",7132,1989,2025,37,199
"""vdem_selected_context""","""V-Dem""","""minimal_context""","""analysis-unit-year""",28092,1789,2025,237,202
"""wdi_selected_context""","""WDI""","""minimal_context""","""analysis-unit-year""",9620,1989,2025,37,260


### What this output tells us

The coverage table shows the observed time span and row volume by source. This is the evidence base for selecting the analysis window later.

The important reading is comparative: UCDP, ACLED, V-Dem and WDI do not all play the same role, so their coverage should inform but not mechanically dictate the final analysis-ready window.

The staging window is intentionally broad: it preserves the observed conflict-source range so that incomplete or partial years remain visible for audit.

The recommended analysis-ready window is stricter: it is derived from the intersection of the primary conflict source and the minimal context sources. ACLED remains a benchmark and therefore does not shrink the primary window.


### Why this cell exists — turn coverage into a window recommendation

The previous table tells us what each source covers. This cell converts that diagnostic into two operational windows.

The staging window is intentionally broad and source-auditable. The recommended analysis-ready window is stricter and is based on the overlap of the primary conflict source and minimal context sources. ACLED remains a benchmark, so it should not shrink the primary analytical window.

In [6]:
conflict_sources=source_coverage.filter(pl.col('family').is_in(['UCDP','ACLED']))
STAGING_YEAR_START=int(conflict_sources.select(pl.col('year_min').min()).item())
STAGING_YEAR_END=int(conflict_sources.select(pl.col('year_max').max()).item())
primary_context_sources=source_coverage.filter(pl.col('source').is_in(['ucdp_organized_violence_cy','vdem_selected_context','wdi_selected_context']))
RECOMMENDED_YEAR_START=int(primary_context_sources.select(pl.col('year_min').max()).item())
RECOMMENDED_YEAR_END=int(primary_context_sources.select(pl.col('year_max').min()).item())

window_diagnostic = pl.DataFrame([
    {"window": "staging_build_window", "year_start": STAGING_YEAR_START, "year_end": STAGING_YEAR_END, "rule": "min/max across conflict sources after coverage diagnostic"},
    {"window": "recommended_analysis_window", "year_start": RECOMMENDED_YEAR_START, "year_end": RECOMMENDED_YEAR_END, "rule": "intersection of UCDP Organized Violence CY, selected V-Dem and selected WDI"},
])

window_diagnostic

window,year_start,year_end,rule
str,i64,i64,str
"""staging_build_window""",1989,2026,"""min/max across conflict sources after coverage diagnostic"""
"""recommended_analysis_window""",1989,2025,"""intersection of UCDP Organized Violence CY, selected V-Dem and selected WDI"""


### What this output tells us

The window diagnostic separates the broad staging range from the stricter recommended analysis range. The staging panel keeps the wider range so partial or out-of-scope years remain auditable.

The analysis-ready window is the first window appropriate for descriptive comparison. If future sources are added, this diagnostic should be rerun rather than hard-coded.

### Why this cell exists — make the window decision visible

The computed dates should not remain hidden inside variables. This cell translates the window diagnostic into a visible note so the reader understands the consequence before any panel rows are built.

In [7]:
display_note(
    f"<strong>Execution insight.</strong> The staging build window is "
    f"<code>{STAGING_YEAR_START}–{STAGING_YEAR_END}</code>. The recommended analysis-ready window is "
    f"<code>{RECOMMENDED_YEAR_START}–{RECOMMENDED_YEAR_END}</code>. "
    "This recommendation is computed from observed coverage and should be revisited if the source set changes."
)

### Consequence for the next step

This visible note turns the computed window into an explicit methodological decision. The next sections can now build a backbone over the staging window while remembering that the final analysis-ready view will be filtered to the recommended range.

<a id="country-crosswalk"></a>

<div class="cl-responsive cl-section-block" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 6</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">Country harmonisation and crosswalk audit</h2>
  <p style='margin:8px 0 0 0;color:#D9E2EC;line-height:1.45;'>Country names are treated as methodological objects, not mere strings.</p>
</div>


Country harmonisation is one of the most consequential parts of this notebook. The raw sources contain country names, historical entities, contested territories and non-country geographic zones. Mapping these labels to ISO3-like panel keys is therefore not a neutral string-cleaning task.

The V1 approach used here follows the decision recorded in the ConflictLens Decisions log: **the 45 unmatched country crosswalk rows are treated as a methodological decision point, not as a technical failure**.

The analysis-unit-year panel remains ISO3-oriented for standard countries, but it also needs a small number of explicitly documented exceptions. ConflictLens therefore distinguishes five situations:

| Case | Treatment in Notebook 02 | Why this matters |
|---|---|---|
| Safe exact match or safe alias | Map to a standard ISO3 code. | These rows can safely join the complete country-year backbone. |
| Validated V1 manual arbitration | Apply the documented project decision. | Manual decisions become reproducible and auditable. |
| Historical or composite entity outside ISO3 | Keep outside the analysis-unit panel and document the analytical unit. | This avoids injecting historical observations into contemporary successor states without evidence. |
| Contested or special statistical entity | Use an explicit analytical code only when the project validates it. | This preserves analytical separability without making a political claim. |
| Maritime or non-country zone | Exclude from the analysis-unit-year panel. | Oceans and maritime zones are not country-year units. |

This design makes the crosswalk file an audit output, not an error dump. In this final V1 build, the 45 initially unmatched rows are no longer treated as open technical failures: they are either mapped where defensible, excluded where outside scope, or retained outside strict ISO3 when forcing a contemporary state would distort the source semantics.

The guiding principle is deliberately strict:

> **Do not silently force ambiguous countries, territories, contested entities or historical entities into ISO3 mappings. Every exception must be explicit, auditable and documented.**


### Why this cell exists — build the country-name audit layer

The raw sources identify places using source-specific country labels. Those labels need to be normalised and mapped to ISO3 before they can be joined to a country-year backbone.

This cell creates the crosswalk inventory. It records which labels matched safely and which labels remain unresolved. The important point is that unresolved labels are **kept as evidence**, not silently dropped without trace.

In [8]:
def raw_country_summary(df:pl.DataFrame, source:str, country_col:str, year_col:str, split_comma=False)->pl.DataFrame:
    work=df.select([pl.col(country_col).alias('raw_country'), pl.col(year_col).cast(pl.Int64, strict=False).alias('year')])
    work=work.with_columns(pl.col('raw_country').cast(pl.Utf8).str.strip_chars())
    if split_comma:
        work=work.with_columns(pl.col('raw_country').str.split(',').alias('raw_country')).explode('raw_country').with_columns(pl.col('raw_country').str.strip_chars())
    work=work.filter(pl.col('raw_country').is_not_null() & (pl.col('raw_country')!=''))
    return work.group_by('raw_country').agg(pl.len().alias('n_rows'), pl.col('year').min().alias('year_min'), pl.col('year').max().alias('year_max')).with_columns(pl.lit(source).alias('source')).select(['source','raw_country','n_rows','year_min','year_max'])

raw_country_inventory=pl.concat([
raw_country_summary(ucdp_ov,'ucdp_organized_violence_cy','country','year'),
raw_country_summary(ucdp_ged,'ucdp_ged_events','country','year'),
raw_country_summary(ucdp_os,'ucdp_one_sided_locations','location','year',split_comma=True),
raw_country_summary(acled_events,'acled_political_violence_events','COUNTRY','YEAR'),
raw_country_summary(acled_fatalities,'acled_reported_fatalities','COUNTRY','YEAR'),
raw_country_summary(acled_civ_events,'acled_civilian_targeting_events','COUNTRY','YEAR'),
raw_country_summary(acled_civ_fatalities,'acled_reported_civilian_fatalities','COUNTRY','YEAR')], how='vertical')
match_rows=[]
for row in raw_country_inventory.iter_rows(named=True):
    match_rows.append(match_country(row['raw_country']))
country_crosswalk=pl.concat([raw_country_inventory, pl.DataFrame(match_rows)], how='horizontal')
if MANUAL_CROSSWALK_PATH.exists():
    manual=pl.read_csv(MANUAL_CROSSWALK_PATH).with_columns([
        pl.col('raw_country').map_elements(normalize_country_name, return_dtype=pl.Utf8).alias('raw_country_norm'),
        pl.col('country_iso3').cast(pl.Utf8).str.to_uppercase().str.strip_chars().alias('manual_iso3')
    ])
    invalid=manual.filter(~pl.col('manual_iso3').is_in(list(VALID_ISO3))).select('manual_iso3').unique().to_series().to_list()
    if invalid: raise ValueError(f'Manual crosswalk contains invalid ISO3 codes: {invalid}')
    country_crosswalk=country_crosswalk.join(manual.select(['raw_country_norm','manual_iso3']), on='raw_country_norm', how='left')
    country_crosswalk=country_crosswalk.with_columns([
        pl.when(pl.col('manual_iso3').is_not_null()).then(pl.col('manual_iso3')).otherwise(pl.col('country_iso3')).alias('country_iso3'),
        pl.when(pl.col('manual_iso3').is_not_null()).then(pl.lit('matched')).otherwise(pl.col('match_status')).alias('match_status'),
        pl.when(pl.col('manual_iso3').is_not_null()).then(pl.lit('manual_crosswalk')).otherwise(pl.col('match_method')).alias('match_method'),
    ]).with_columns(pl.col('country_iso3').map_elements(iso3_to_name, return_dtype=pl.Utf8).alias('country_name')).drop('manual_iso3')

crosswalk_summary = (
    country_crosswalk
    .group_by(["source", "match_status", "match_method"])
    .agg(
        pl.col("raw_country").n_unique().alias("raw_country_values"),
        pl.col("n_rows").sum().alias("source_rows"),
    )
    .sort(["source", "match_status", "match_method"])
)

crosswalk_summary

source,match_status,match_method,raw_country_values,source_rows
str,str,str,u32,u32
"""acled_civilian_targeting_events""","""ambiguous""","""blocked_ambiguous_name""",1,9
"""acled_civilian_targeting_events""","""matched""","""pycountry_exact""",224,2521
"""acled_civilian_targeting_events""","""matched""","""safe_alias""",18,160
"""acled_civilian_targeting_events""","""unmatched""","""no_conservative_match""",7,27
"""acled_political_violence_events""","""ambiguous""","""blocked_ambiguous_name""",1,9
"""acled_political_violence_events""","""matched""","""pycountry_exact""",224,2546
"""acled_political_violence_events""","""matched""","""safe_alias""",18,165
"""acled_political_violence_events""","""unmatched""","""no_conservative_match""",7,34
"""acled_reported_civilian_fatalities""","""ambiguous""","""blocked_ambiguous_name""",1,9


### What this output tells us

The crosswalk inventory is the first point where country harmonisation becomes auditable. Matched labels can be joined to the panel; unresolved labels remain visible with their source, row count and year range.

This output is deliberately conservative. An unmatched row is not automatically a bug: it often signals a real methodological decision, such as how to treat historical states or contested territories.

### Why this cell exists — apply the validated V1 country arbitration

The previous cell deliberately stops at conservative automatic matching. The remaining unmatched rows were reviewed as a project-level methodological decision rather than as a string-matching problem.

This cell encodes the validated V1 arbitration directly in the notebook. The arbitration table is the executable version of the methodological decision: it tells the pipeline which labels can be mapped, which labels must be excluded, and which labels are resolved but intentionally kept outside strict ISO3.

| Outcome | Meaning for the analysis-unit-year panel |
|---|---|
| `map` | The source label is mapped to a defensible standard ISO3 target. |
| `map_special_statistical_unit` | The source label is mapped to a validated analytical code, currently `XKX` for Kosovo. |
| `exclude` | The row is outside the country-year analytical unit, usually a maritime zone or non-country territory. |
| `keep_historical_entity` | The source label is resolved as a historical unit and is not forced into a contemporary successor state. |
| `keep_historical_composite_entity` | The source label is resolved as a composite historical unit whose observations should not be attributed to a single contemporary state. |

The most important point is that historical and composite entities are not silently dropped as accidental unmatched rows. They are resolved as **outside strict ISO3** and preserved in the arbitration export for auditability.


### Validated V1 arbitration policy — detailed methodological documentation

The arbitration below is the V1 policy validated for ConflictLens. It is intentionally verbose because country harmonisation affects the interpretation of every downstream country-year indicator.

| Raw country | Sources concerned | V1 action | Target code / unit | Rationale | Impact note |
|---|---|---|---|---|---|
| `Indian Ocean` | ACLED | `exclude` | — | Maritime zone, not a country-year unit. | Low ACLED-only impact. |
| `Mediterranean Sea` | ACLED | `exclude` | — | Maritime zone, not a country-year unit. | Low ACLED-only impact. |
| `Atlantic Ocean` | ACLED | `exclude` | — | Maritime zone, not a country-year unit. | Low ACLED-only impact. |
| `Pacific Ocean` | ACLED | `exclude` | — | Maritime zone, not a country-year unit. | Low ACLED-only impact. |
| `Southern Ocean` | ACLED | `exclude` | — | Maritime zone, not a country-year unit. | Very low ACLED-only impact. |
| `Akrotiri and Dhekelia` | ACLED | `exclude` | — | Territory / sovereign base area. Mapping to Cyprus or the United Kingdom would be more misleading than excluding it for V1. | Low ACLED-only impact. |
| `French Southern and Antarctic Lands` | ACLED | `map` | `ATF` | Technical territory alias with a defensible alpha-3 target. | Negligible impact, but cleanly resolves a technical unmatched case. |
| `German Democratic Republic` | UCDP | `documented_exclusion_outside_iso3` | `HIST_GDR` | Do not inject 1989–1990 East Germany observations into contemporary Germany. The historical transition is clear, but statistical continuity should not be assumed silently. | Very low UCDP impact. |
| `Yemen (South Yemen)` | UCDP | `documented_exclusion_outside_iso3` | `HIST_SOUTH_YEMEN` | South Yemen is documented as a separate historical unit before unification and excluded from the analysis-unit panel with ISO3 as an optional attribute. | Very low UCDP impact. |
| `Czechoslovakia` | UCDP | `documented_exclusion_outside_iso3` | `HIST_CZECHOSLOVAKIA` | No unique successor state can be selected without arbitrary allocation between Czechia and Slovakia. | Low UCDP impact. |
| `Kosovo` | UCDP + ACLED | `map_special_statistical_unit` | `XKX` | Kosovo is treated as an independent statistical unit for analytical consistency. This is not a political recognition claim. | Moderate methodological importance, limited row volume. |
| `Yemen (North Yemen)` | UCDP | `temporal_split` | `HIST_NORTH_YEMEN` before unified-Yemen coverage; `YEM` for unified-Yemen coverage | The former North Yemen and unified Yemen are two different statistical entities. UCDP GED rows run from 1994 to 2025 and therefore correspond to unified Yemen despite the source label. Pre-unification North Yemen is documented outside the analysis-unit panel. | Very high UCDP impact, especially GED. |
| `Russia (Soviet Union)` | UCDP | `map_with_caveat` | `RUS` | Pragmatic successor-state / central-state convention. The political center of the Soviet Union was Moscow, and the Soviet-era share of the data is statistically limited. This combination justifies `RUS` for V1. If the dataset contained decades of Soviet-period observations, this decision would be different. | Very high UCDP impact overall; limited early Soviet-period share. |
| `Serbia (Yugoslavia)` | UCDP | `documented_exclusion_outside_iso3` | `HIST_SERBIA_YUGOSLAVIA` or future period-specific historical units | Do not map Yugoslav-war observations to contemporary Serbia. Conflicts from the 1990s are historically composite and should not be attributed to Serbia alone. | High UCDP impact and important historical interpretation risk. |

The table above should be read as a **statistical decision table**, not as a political statement. Its purpose is to protect the consistency of the analytical unit used by the panel. Historical/composite identifiers are retained in the audit table, but they are **not** added as rows to the strict `analysis_unit_id + year` analysis-ready panel.


### Case-by-case methodological rules retained for V1

#### ACLED maritime zones

`Indian Ocean`, `Mediterranean Sea`, `Atlantic Ocean`, `Pacific Ocean` and `Southern Ocean` are excluded. They are locations in the ACLED source data, but they are not country-year units. Mapping them to nearby coastal states would create arbitrary attribution.

This exclusion has low analytical impact because these rows are ACLED-only and low-volume. The benefit is methodological clarity: the analysis-unit-year panel should not contain ocean-level observations disguised as countries.

#### Akrotiri and Dhekelia

`Akrotiri and Dhekelia` is excluded for V1. It is a sovereign base area / territory, not a standard country-year unit in the panel. Mapping it to Cyprus or to the United Kingdom would each imply a different territorial convention. Because neither mapping is necessary for the V1 analysis and the observed impact is low, exclusion is the safest decision.

#### French Southern and Antarctic Lands

`French Southern and Antarctic Lands` is mapped to `ATF`. This is treated as a technical territory alias rather than a major geopolitical arbitration. The impact is negligible, but mapping it removes an avoidable technical unmatched case.

#### Small historical entities

`German Democratic Republic`, `Yemen (South Yemen)` and `Czechoslovakia` are **documented as outside the analysis-unit panel with ISO3 as an optional attribute**. Their historical identifiers remain in the arbitration table for traceability, but no `HIST_*` rows are inserted into `analysis_ready`.

This is a deliberate statistical choice. Even when a historical transition is clear, forcing the observation into a contemporary successor state can create false comparability. For example, a `DEU-1989` row containing East Germany is not semantically identical to a later unified-Germany row. Similarly, Czechoslovakia cannot be assigned to either Czechia or Slovakia without an arbitrary allocation rule.

#### Kosovo

Kosovo is represented by `XKX` as a ConflictLens statistical unit.

This does not express a political position on status or recognition. The choice only defines the analytical unit used by the panel. Statistically, Kosovo-specific observations should not be hidden inside Serbia, and keeping them separate makes the crosswalk easier to audit.

#### Yemen / North Yemen

ConflictLens treats North Yemen and unified Yemen as distinct statistical entities.

The source label `Yemen (North Yemen)` is not sufficient by itself to decide the mapping. The year matters:

- pre-unification observations are documented outside the analysis-unit panel as `HIST_NORTH_YEMEN`;
- observations corresponding to the unified Yemeni state map to `YEM`;
- because the analysis-unit-year grain cannot split the exact date of 22 May 1990 cleanly, the 1990 boundary must be handled conservatively and documented.

The empirical check from the UCDP data showed that the high-impact GED observations for `Yemen (North Yemen)` run from 1994 to 2025. These are interpreted as unified Yemen observations under a legacy source label and are therefore mapped to `YEM`.

#### Russia / Soviet Union

`Russia (Soviet Union)` is mapped to `RUS` for V1, but this is explicitly not a pure territorial attribution.

The rationale combines two arguments:

1. **Statistical argument:** the 1989–1992 portion is limited relative to the full `Russia (Soviet Union)` series.
2. **Historical-institutional argument:** the political center of the Soviet Union was Moscow, which makes `RUS` a defensible successor-state / central-state convention for a V1 analysis-unit-year panel.

This decision depends on both arguments together. If the dataset contained a long Soviet-period time series, ConflictLens would not map the whole entity to `RUS` merely because Moscow was the political center.

#### Serbia / Yugoslavia

`Serbia (Yugoslavia)` is not mapped to contemporary `SRB` for V1.

The 1990s conflicts attached to this label should not be interpreted as events belonging to Serbia alone. The source label includes Yugoslav and Serbia/Montenegro-era complexity, and the analysis-unit-year panel should not collapse these conflicts into contemporary Serbia without a much more detailed event-level attribution strategy.

For V1, `Serbia (Yugoslavia)` is documented as an outside-ISO3 historical/composite unit and excluded from the strict contemporary ISO3 analysis-ready panel. A future event-level notebook may split this case more precisely, for example into:

- `HIST_YUGOSLAVIA` for the Yugoslav period;
- `HIST_SERBIA_MONTENEGRO` for the Serbia and Montenegro period;
- `SRB` only for observations that can be cleanly attributed to Serbia.

This is the most important historical exclusion to keep visible in Notebook 03.


In [9]:
ACLED_SOURCES = [
    "acled_political_violence_events",
    "acled_reported_fatalities",
    "acled_civilian_targeting_events",
    "acled_reported_civilian_fatalities",
]

UCDP_SOURCES = [
    "ucdp_organized_violence_cy",
    "ucdp_ged_events",
    "ucdp_one_sided_locations",
]

arbitration_rows: list[dict[str, Any]] = []

def add_arbitration(
    sources: list[str],
    raw_country: str,
    category: str,
    proposed_action: str,
    target_iso3: str | None,
    target_country_name: str | None,
    rationale: str,
    confidence: str,
    impact_note: str,
    requires_user_validation: bool,
    decision_year_min: int | None = None,
    decision_year_max: int | None = None,
    geo_unit_id: str | None = None,
    geo_unit_type: str | None = None,
) -> None:
    for source in sources:
        arbitration_rows.append({
            "source": source,
            "raw_country": raw_country,
            "decision_year_min": decision_year_min,
            "decision_year_max": decision_year_max,
            "category": category,
            "proposed_action": proposed_action,
            "target_iso3": target_iso3,
            "target_country_name": target_country_name,
            "geo_unit_id": geo_unit_id,
            "geo_unit_type": geo_unit_type,
            "rationale": rationale,
            "confidence": confidence,
            "impact_note": impact_note,
            "requires_user_validation": requires_user_validation,
        })

for maritime_zone in ["Indian Ocean", "Mediterranean Sea", "Atlantic Ocean", "Pacific Ocean", "Southern Ocean"]:
    add_arbitration(
        ACLED_SOURCES,
        maritime_zone,
        "maritime_non_country",
        "exclude",
        None,
        None,
        "Maritime zone, not a analysis-unit-year analytical unit.",
        "high",
        "Low ACLED-only impact; excluding avoids arbitrary country attribution.",
        False,
    )

add_arbitration(
    ACLED_SOURCES,
    "Akrotiri and Dhekelia",
    "territory",
    "exclude",
    None,
    None,
    "Sovereign base area / territory. Mapping to Cyprus or the United Kingdom would be more misleading than excluding it for V1.",
    "high",
    "Low ACLED-only impact.",
    False,
)

add_arbitration(
    ACLED_SOURCES,
    "French Southern and Antarctic Lands",
    "territory",
    "map",
    "ATF",
    "French Southern Territories",
    "Technical territory alias with a defensible alpha-3 target.",
    "high",
    "Negligible impact, but cleanly resolves a technical unmatched case.",
    False,
)

add_arbitration(
    ["ucdp_organized_violence_cy"],
    "German Democratic Republic",
    "historical_entity",
    "keep_as_non_iso_analysis_unit",
    None,
    "German Democratic Republic",
    "East Germany is not assigned to contemporary Germany. It is retained as a non-ISO historical analysis unit so the violence signal remains available without inventing an ISO3 mapping.",
    "high",
    "Very low UCDP impact, but retained for methodological consistency.",
    False,
    1989,
    1990,
    "HIST_GDR",
    "historical_entity",
)

add_arbitration(
    ["ucdp_organized_violence_cy"],
    "Yemen (South Yemen)",
    "historical_entity",
    "keep_as_non_iso_analysis_unit",
    None,
    "Yemen (South Yemen)",
    "South Yemen is retained as a separate pre-unification non-ISO analysis unit instead of being merged into unified Yemen.",
    "high",
    "Very low UCDP impact, but retained for methodological consistency.",
    False,
    1989,
    1990,
    "HIST_SOUTH_YEMEN",
    "historical_entity",
)

add_arbitration(
    ["ucdp_organized_violence_cy"],
    "Czechoslovakia",
    "historical_entity",
    "keep_as_non_iso_analysis_unit",
    None,
    "Czechoslovakia",
    "No unique successor state can be selected without arbitrary allocation between Czechia and Slovakia; the source label is retained as a non-ISO historical analysis unit.",
    "high",
    "Low UCDP impact, but retained for methodological consistency.",
    False,
    1989,
    1992,
    "HIST_CZECHOSLOVAKIA",
    "historical_entity",
)

add_arbitration(
    ["ucdp_organized_violence_cy", *ACLED_SOURCES],
    "Kosovo",
    "contested_entity",
    "map_special_statistical_unit",
    "XKX",
    "Kosovo",
    "Kosovo is treated as an independent statistical unit for analytical consistency. This is not a political recognition claim.",
    "high",
    "Limited row volume, but important methodological clarity.",
    True,
)

add_arbitration(
    ["ucdp_organized_violence_cy"],
    "Yemen (North Yemen)",
    "historical_entity",
    "keep_as_non_iso_analysis_unit",
    None,
    "Yemen (North Yemen)",
    "North Yemen before unification is retained as a non-ISO historical analysis unit. At analysis-unit-year grain, 1990 is treated conservatively as pre-unified for this label.",
    "high",
    "Pre-unification rows have low direct violence impact but matter for semantic integrity and are retained as a non-ISO unit.",
    True,
    1989,
    1990,
    "HIST_NORTH_YEMEN",
    "historical_entity",
)

add_arbitration(
    ["ucdp_organized_violence_cy"],
    "Yemen (North Yemen)",
    "current_state_alias_after_unification",
    "map",
    "YEM",
    "Yemen",
    "From 1991 onward this label is treated as unified Yemen for the analysis-unit-year panel.",
    "high",
    "Important for preserving unified Yemen coverage in UCDP Organized Violence CY.",
    True,
    1991,
    2025,
)

add_arbitration(
    ["ucdp_ged_events", "ucdp_one_sided_locations"],
    "Yemen (North Yemen)",
    "current_state_alias_after_unification",
    "map",
    "YEM",
    "Yemen",
    "The observed rows for this label in these sources occur after unification and are treated as unified Yemen under a legacy source label.",
    "high",
    "Very high UCDP GED impact; mapping avoids losing unified Yemen events.",
    True,
    1991,
    2025,
)

add_arbitration(
    UCDP_SOURCES,
    "Russia (Soviet Union)",
    "historical_successor_convention",
    "map",
    "RUS",
    "Russian Federation",
    "Mapped to RUS as a pragmatic successor-state / central-state convention. The decision depends on both the limited Soviet-period share and Moscow's role as the political center of the USSR; a long Soviet-period series would require a separate historical USSR unit.",
    "medium",
    "Very high UCDP impact overall; limited early Soviet-period share.",
    True,
)

add_arbitration(
    UCDP_SOURCES,
    "Serbia (Yugoslavia)",
    "historical_composite_entity",
    "keep_as_non_iso_analysis_unit",
    None,
    "Serbia (Yugoslavia)",
    "Yugoslav-war and Serbia/Montenegro-era observations are retained as a non-ISO historical composite analysis unit rather than attributed to Serbia alone.",
    "high",
    "High UCDP impact; retention as a non-ISO unit avoids both data loss and arbitrary remapping.",
    True,
    None,
    None,
    "HIST_SERBIA_YUGOSLAVIA",
    "historical_composite_entity",
)

country_crosswalk_arbitration_v1 = (
    pl.DataFrame(arbitration_rows)
    .with_columns([
        pl.col("decision_year_min").cast(pl.Int64, strict=False),
        pl.col("decision_year_max").cast(pl.Int64, strict=False),
        pl.col("requires_user_validation").cast(pl.Boolean),
    ])
    .join(
        country_crosswalk.select(["source", "raw_country", "n_rows", "year_min", "year_max"]),
        on=["source", "raw_country"],
        how="left",
    )
    .filter(pl.col("n_rows").is_not_null())
    .select([
        "source", "raw_country", "n_rows", "year_min", "year_max",
        "decision_year_min", "decision_year_max", "category", "proposed_action",
        "target_iso3", "target_country_name", "geo_unit_id", "geo_unit_type",
        "rationale", "confidence", "impact_note", "requires_user_validation",
    ])
    .sort(["source", "raw_country", "decision_year_min", "decision_year_max"], nulls_last=True)
)

invalid_arbitration_codes = (
    country_crosswalk_arbitration_v1
    .filter(pl.col("target_iso3").is_not_null() & ~pl.col("target_iso3").is_in(list(VALID_ISO3)))
    .select("target_iso3")
    .unique()
    .to_series()
    .to_list()
)
if invalid_arbitration_codes:
    raise ValueError(f"V1 crosswalk arbitration contains invalid target codes: {invalid_arbitration_codes}")

country_crosswalk_arbitration_v1.head(50)

source,raw_country,n_rows,year_min,year_max,decision_year_min,decision_year_max,category,proposed_action,target_iso3,target_country_name,geo_unit_id,geo_unit_type,rationale,confidence,impact_note,requires_user_validation
str,str,u32,i64,i64,i64,i64,str,str,str,str,str,str,str,str,str,bool
"""acled_civilian_targeting_events""","""Akrotiri and Dhekelia""",4,2018,2021,null,null,"""territory""","""exclude""",null,null,null,null,"""Sovereign base area / territory. Mapping to Cyprus or the United Kingdom would be more misleading than excluding it for …","""high""","""Low ACLED-only impact.""",false
"""acled_civilian_targeting_events""","""Atlantic Ocean""",1,2020,2020,null,null,"""maritime_non_country""","""exclude""",null,null,null,null,"""Maritime zone, not a analysis-unit-year analytical unit.""","""high""","""Low ACLED-only impact; excluding avoids arbitrary country attribution.""",false
"""acled_civilian_targeting_events""","""French Southern and Antarctic Lands""",1,2021,2021,null,null,"""territory""","""map""","""ATF""","""French Southern Territories""",null,null,"""Technical territory alias with a defensible alpha-3 target.""","""high""","""Negligible impact, but cleanly resolves a technical unmatched case.""",false
"""acled_civilian_targeting_events""","""Indian Ocean""",12,2015,2026,null,null,"""maritime_non_country""","""exclude""",null,null,null,null,"""Maritime zone, not a analysis-unit-year analytical unit.""","""high""","""Low ACLED-only impact; excluding avoids arbitrary country attribution.""",false
"""acled_civilian_targeting_events""","""Kosovo""",9,2018,2026,null,null,"""contested_entity""","""map_special_statistical_unit""","""XKX""","""Kosovo""",null,null,"""Kosovo is treated as an independent statistical unit for analytical consistency. This is not a political recognition cla…","""high""","""Limited row volume, but important methodological clarity.""",true
"""acled_civilian_targeting_events""","""Mediterranean Sea""",5,2022,2026,null,null,"""maritime_non_country""","""exclude""",null,null,null,null,"""Maritime zone, not a analysis-unit-year analytical unit.""","""high""","""Low ACLED-only impact; excluding avoids arbitrary country attribution.""",false
"""acled_civilian_targeting_events""","""Pacific Ocean""",3,2019,2026,null,null,"""maritime_non_country""","""exclude""",null,null,null,null,"""Maritime zone, not a analysis-unit-year analytical unit.""","""high""","""Low ACLED-only impact; excluding avoids arbitrary country attribution.""",false
"""acled_civilian_targeting_events""","""Southern Ocean""",1,2026,2026,null,null,"""maritime_non_country""","""exclude""",null,null,null,null,"""Maritime zone, not a analysis-unit-year analytical unit.""","""high""","""Low ACLED-only impact; excluding avoids arbitrary country attribution.""",false
"""acled_political_violence_events""","""Akrotiri and Dhekelia""",5,2018,2026,null,null,"""territory""","""exclude""",null,null,null,null,"""Sovereign base area / territory. Mapping to Cyprus or the United Kingdom would be more misleading than excluding it for …","""high""","""Low ACLED-only impact.""",false


### What this output tells us

The arbitration table is now part of the reproducible build. It is source-specific and, where needed, year-specific.

The output should be read as the machine-readable counterpart of the methodological policy documented above. Each row records:

- the source affected by the decision;
- the raw country label being arbitrated;
- the observed row volume and year range;
- the V1 action selected by the project;
- the target ISO3 or special analytical code when a mapping is accepted;
- the historical or composite unit identifier when the case is documented outside the analysis-unit panel with ISO3 as an optional attribute;
- the rationale, confidence level and validation flag.

The most important split is `Yemen (North Yemen)`: pre-unification observations remain outside strict ISO3, while post-unification observations are mapped to `YEM`. `Russia (Soviet Union)` is mapped to `RUS` with a strong documented caveat. `Serbia (Yugoslavia)` is intentionally documented outside strict ISO3 and excluded from the analysis-ready panel because assigning Yugoslav-war observations to contemporary Serbia would be historically misleading. `Kosovo` is represented by `XKX` as a statistical unit, not as a political recognition claim.


In [10]:
decision_summary = (
    country_crosswalk_arbitration_v1
    .group_by(["source", "raw_country"])
    .agg([
        pl.col("proposed_action").unique().alias("v1_actions"),
        pl.col("category").unique().alias("v1_categories"),
        pl.col("target_iso3").drop_nulls().unique().alias("v1_target_iso3"),
        pl.col("geo_unit_id").drop_nulls().unique().alias("v1_geo_unit_id"),
        pl.col("target_iso3").is_not_null().any().alias("v1_has_target_iso3"),
        ((pl.col("geo_unit_id").is_not_null()) & (pl.col("target_iso3").is_null()) & (pl.col("proposed_action") == "keep_as_non_iso_analysis_unit")).any().alias("v1_has_non_iso_analysis_unit"),
        (pl.col("proposed_action") == "exclude").any().alias("v1_has_exclusion"),
    ])
)

country_crosswalk_resolution = (
    country_crosswalk
    .join(decision_summary, on=["source", "raw_country"], how="left")
    .with_columns([
        pl.when(pl.col("country_iso3").is_not_null())
        .then(pl.lit("auto_matched"))
        .when(pl.col("v1_has_target_iso3").fill_null(False) & pl.col("v1_has_non_iso_analysis_unit").fill_null(False))
        .then(pl.lit("partially_mapped_and_kept_as_non_iso_unit_by_arbitration"))
        .when(pl.col("v1_has_target_iso3").fill_null(False))
        .then(pl.lit("mapped_by_arbitration"))
        .when(pl.col("v1_has_non_iso_analysis_unit").fill_null(False))
        .then(pl.lit("kept_as_non_iso_analysis_unit_by_arbitration"))
        .when(pl.col("v1_has_exclusion").fill_null(False))
        .then(pl.lit("excluded_by_arbitration"))
        .otherwise(pl.lit("unresolved_after_arbitration"))
        .alias("resolution_status")
    ])
)

country_crosswalk_resolution_summary = (
    country_crosswalk_resolution
    .group_by(["source", "resolution_status"])
    .agg([
        pl.col("raw_country").n_unique().alias("raw_country_values"),
        pl.col("n_rows").sum().alias("source_rows"),
    ])
    .sort(["source", "resolution_status"])
)

country_crosswalk_resolution_summary

source,resolution_status,raw_country_values,source_rows
str,str,u32,u32
"""acled_civilian_targeting_events""","""auto_matched""",242,2681
"""acled_civilian_targeting_events""","""excluded_by_arbitration""",6,26
"""acled_civilian_targeting_events""","""mapped_by_arbitration""",2,10
"""acled_political_violence_events""","""auto_matched""",242,2711
"""acled_political_violence_events""","""excluded_by_arbitration""",6,33
"""acled_political_violence_events""","""mapped_by_arbitration""",2,10
"""acled_reported_civilian_fatalities""","""auto_matched""",242,2681
"""acled_reported_civilian_fatalities""","""excluded_by_arbitration""",6,26
"""acled_reported_civilian_fatalities""","""mapped_by_arbitration""",2,10


The next table now shows only labels that remain unresolved **after** the validated V1 arbitration. If the arbitration table covers all 45 initially unmatched rows, this table should be empty.

An empty table does **not** mean that every row has been forced into the ISO3 panel. It means that every initially unmatched label now has an explicit status: mapped, excluded, mapped to a special statistical unit, or retained as a documented historical/composite entity outside strict ISO3.


### Why this cell exists — inspect remaining unresolved labels after arbitration

A country crosswalk can be publication-grade only if its residual ambiguity is explicit. This cell verifies that the previously unmatched rows have either been mapped, excluded or resolved outside strict ISO3 through the V1 arbitration table.

Rows that are historical or excluded by decision should not appear here: they are no longer unresolved technical failures.

In [11]:
country_crosswalk_unmatched = (
    country_crosswalk_resolution
    .filter(pl.col("resolution_status") == "unresolved_after_arbitration")
    .select([
        "source", "raw_country", "n_rows", "year_min", "year_max",
        "raw_country_norm", "country_iso3", "country_name", "match_status",
        "match_method", "candidate_matches", "resolution_status",
    ])
    .sort(["n_rows", "source", "raw_country"], descending=[True, False, False])
)

country_crosswalk_unmatched.head(30)

source,raw_country,n_rows,year_min,year_max,raw_country_norm,country_iso3,country_name,match_status,match_method,candidate_matches,resolution_status
str,str,u32,i64,i64,str,str,str,str,str,str,str


### What this output tells us

The remaining unresolved table is the final crosswalk quality gate. Empty output means the 45 initially unmatched rows have all received an explicit V1 decision.

This does not mean every row was mapped into the analysis-unit panel. Some rows are deliberately excluded, and some historical or composite entities are deliberately retained outside strict ISO3. The point is that none of these outcomes is silent.

For downstream notebooks, this distinction matters:

- mapped rows can participate in the standard `analysis_unit_id + year` panel;
- special statistical units such as `XKX` can be used analytically with explicit documentation;
- historical/composite units should be handled as audit or extension-layer cases unless a later notebook defines a stronger attribution rule;
- excluded maritime or non-country zones should not appear in the country-year backbone.


### Why this cell exists — persist crosswalk arbitration artefacts

The crosswalk arbitration is now a durable audit artefact. The notebook writes both the V1 decision table and the remaining-unresolved diagnostic.

`02_country_crosswalk_arbitration_v1.csv` documents the project decision. It should be treated as the source of truth for the V1 country arbitration policy used by Notebook 02.

`02_country_crosswalk_unmatched.csv` documents any residual unresolved labels after that decision has been applied. In the validated V1 build, this file is expected to be empty because all 45 initially unmatched rows have been explicitly handled.

Persisting both files is important for handoff. Notebook 03 should not have to rediscover why a country label was mapped, excluded or retained as non-ISO analysis units; it should be able to read the exported arbitration trail.


In [12]:
crosswalk_arbitration_path = OUTPUT_DIR / "02_country_crosswalk_arbitration_v1.csv"
crosswalk_unmatched_path = OUTPUT_DIR / "02_country_crosswalk_unmatched.csv"
country_crosswalk_arbitration_v1.write_csv(crosswalk_arbitration_path)
country_crosswalk_unmatched.write_csv(crosswalk_unmatched_path)

resolved_values = int(country_crosswalk_resolution.filter(pl.col("resolution_status") != "unresolved_after_arbitration").height)
unresolved_values = int(country_crosswalk_unmatched.height)
display_note(
    f"<strong>Execution insight.</strong> V1 crosswalk arbitration exported to "
    f"<code>{crosswalk_arbitration_path.relative_to(PROJECT_ROOT)}</code>. "
    f"Remaining unresolved diagnostic exported to <code>{crosswalk_unmatched_path.relative_to(PROJECT_ROOT)}</code>. "
    f"Resolved or intentionally handled source-country rows: <code>{resolved_values}</code>; "
    f"remaining unresolved source-country rows: <code>{unresolved_values}</code>. "
    "Historical and composite entities that cannot legitimately receive an ISO3 code are retained as non-ISO analysis units rather than force-mapped or dropped."
)

### Why this cell exists — quantify the impact of arbitration decisions

A crosswalk decision is not complete until its empirical footprint is visible. This report summarises the row volume, year range and available fatality metric for every V1 arbitration rule. It is especially important for outside-ISO3 decisions such as `Serbia (Yugoslavia)`, where retention as a non-ISO analysis unit is methodologically necessary and analytically non-trivial.


In [13]:
# Crosswalk arbitration impact report
#
# This report makes visible the empirical footprint of V1 arbitration decisions.
# Non-ISO historical/composite cases are retained as analysis units, not dropped.

def _impact_by_raw_country(df: pl.DataFrame, source: str, raw_col: str, year_col: str, metric_exprs: list[pl.Expr]) -> pl.DataFrame:
    return (
        df
        .with_columns([
            pl.col(raw_col).cast(pl.Utf8).str.strip_chars().alias("raw_country"),
            pl.col(year_col).cast(pl.Int64, strict=False).alias("year"),
        ])
        .group_by("raw_country")
        .agg([
            pl.len().alias("raw_rows_recomputed"),
            pl.col("year").min().alias("impact_year_min"),
            pl.col("year").max().alias("impact_year_max"),
            *metric_exprs,
        ])
        .with_columns(pl.lit(source).alias("source"))
    )

ucdp_ged_impact = _impact_by_raw_country(
    ucdp_ged,
    "ucdp_ged_events",
    "country",
    "year",
    [pl.col("best").cast(pl.Float64, strict=False).sum().alias("fatalities_best_affected")],
)

ucdp_ov_impact = _impact_by_raw_country(
    ucdp_ov,
    "ucdp_organized_violence_cy",
    "country",
    "year",
    [pl.lit(None).cast(pl.Float64).alias("fatalities_best_affected")],
)

ucdp_os_impact_work = (
    ucdp_os
    .with_columns([
        pl.col("location").cast(pl.Utf8).str.split(",").alias("location_components"),
        pl.col("year").cast(pl.Int64, strict=False),
    ])
    .with_columns(pl.col("location_components").list.len().alias("location_count"))
    .explode("location_components")
    .rename({"location_components": "raw_country"})
    .with_columns([
        pl.col("raw_country").cast(pl.Utf8).str.strip_chars(),
        (pl.col("best_fatality_estimate").cast(pl.Float64, strict=False) / pl.col("location_count")).alias("best_fatality_estimate_allocated"),
    ])
)
ucdp_os_impact = _impact_by_raw_country(
    ucdp_os_impact_work,
    "ucdp_one_sided_locations",
    "raw_country",
    "year",
    [pl.col("best_fatality_estimate_allocated").sum().alias("fatalities_best_affected")],
)

acled_impact_frames = []
for source_name, raw_df, value_col in [
    ("acled_political_violence_events", acled_events, None),
    ("acled_reported_fatalities", acled_fatalities, "FATALITIES"),
    ("acled_civilian_targeting_events", acled_civ_events, None),
    ("acled_reported_civilian_fatalities", acled_civ_fatalities, "FATALITIES"),
]:
    metric = (
        pl.col(value_col).cast(pl.Float64, strict=False).sum().alias("fatalities_best_affected")
        if value_col
        else pl.lit(None).cast(pl.Float64).alias("fatalities_best_affected")
    )
    acled_impact_frames.append(_impact_by_raw_country(raw_df, source_name, "COUNTRY", "YEAR", [metric]))

arbitration_impact_metrics = pl.concat([ucdp_ov_impact, ucdp_ged_impact, ucdp_os_impact, *acled_impact_frames], how="diagonal")

crosswalk_arbitration_impact_report = (
    country_crosswalk_arbitration_v1
    .join(arbitration_impact_metrics, on=["source", "raw_country"], how="left")
    .with_columns([
        (pl.col("proposed_action") == "keep_as_non_iso_analysis_unit").alias("kept_as_non_iso_analysis_unit"),
        (pl.col("proposed_action") == "exclude").alias("excluded_from_analysis_unit_panel"),
        pl.when(pl.col("target_iso3").is_not_null())
        .then(pl.col("target_iso3"))
        .when(pl.col("geo_unit_id").is_not_null())
        .then(pl.col("geo_unit_id"))
        .otherwise(None)
        .alias("analysis_unit_id"),
        pl.when(pl.col("target_country_name").is_not_null())
        .then(pl.col("target_country_name"))
        .when(pl.col("geo_unit_id").is_not_null())
        .then(pl.col("raw_country"))
        .otherwise(None)
        .alias("analysis_unit_label"),
        pl.when(pl.col("source") == "ucdp_organized_violence_cy")
        .then(pl.lit("No additive fatality metric used here; UCDP OV contains cumulative analysis-unit-year fields."))
        .when(pl.col("source") == "ucdp_ged_events")
        .then(pl.lit("Sum of GED best fatalities for affected raw country label."))
        .when(pl.col("source") == "ucdp_one_sided_locations")
        .then(pl.lit("Sum of allocated UCDP One-sided best fatalities for affected location label."))
        .when(pl.col("source").str.contains("fatalities"))
        .then(pl.lit("Sum of ACLED reported fatalities for affected raw country label."))
        .otherwise(pl.lit("No fatality metric for this source table."))
        .alias("fatality_metric_note"),
    ])
    .select([
        "source", "raw_country", "proposed_action", "category", "target_iso3", "target_country_name",
        "geo_unit_id", "geo_unit_type", "analysis_unit_id", "analysis_unit_label",
        "kept_as_non_iso_analysis_unit", "excluded_from_analysis_unit_panel",
        "n_rows", "year_min", "year_max", "raw_rows_recomputed", "impact_year_min", "impact_year_max",
        "fatalities_best_affected", "fatality_metric_note", "rationale", "impact_note", "confidence"
    ])
    .sort(["kept_as_non_iso_analysis_unit", "excluded_from_analysis_unit_panel", "source", "raw_country"], descending=[True, True, False, False])
)

crosswalk_arbitration_impact_report

source,raw_country,proposed_action,category,target_iso3,target_country_name,geo_unit_id,geo_unit_type,analysis_unit_id,analysis_unit_label,kept_as_non_iso_analysis_unit,excluded_from_analysis_unit_panel,n_rows,year_min,year_max,raw_rows_recomputed,impact_year_min,impact_year_max,fatalities_best_affected,fatality_metric_note,rationale,impact_note,confidence
str,str,str,str,str,str,str,str,str,str,bool,bool,u32,i64,i64,u32,i64,i64,f64,str,str,str,str
"""ucdp_ged_events""","""Serbia (Yugoslavia)""","""keep_as_non_iso_analysis_unit""","""historical_composite_entity""",null,"""Serbia (Yugoslavia)""","""HIST_SERBIA_YUGOSLAVIA""","""historical_composite_entity""","""HIST_SERBIA_YUGOSLAVIA""","""Serbia (Yugoslavia)""",true,false,1204,1991,2001,1204,1991,2001,8773.0,"""Sum of GED best fatalities for affected raw country label.""","""Yugoslav-war and Serbia/Montenegro-era observations are retained as a non-ISO historical composite analysis unit rather …","""High UCDP impact; retention as a non-ISO unit avoids both data loss and arbitrary remapping.""","""high"""
"""ucdp_one_sided_locations""","""Serbia (Yugoslavia)""","""keep_as_non_iso_analysis_unit""","""historical_composite_entity""",null,"""Serbia (Yugoslavia)""","""HIST_SERBIA_YUGOSLAVIA""","""historical_composite_entity""","""HIST_SERBIA_YUGOSLAVIA""","""Serbia (Yugoslavia)""",true,false,5,1992,1999,5,1992,1999,6969.0,"""Sum of allocated UCDP One-sided best fatalities for affected location label.""","""Yugoslav-war and Serbia/Montenegro-era observations are retained as a non-ISO historical composite analysis unit rather …","""High UCDP impact; retention as a non-ISO unit avoids both data loss and arbitrary remapping.""","""high"""
"""ucdp_organized_violence_cy""","""Czechoslovakia""","""keep_as_non_iso_analysis_unit""","""historical_entity""",null,"""Czechoslovakia""","""HIST_CZECHOSLOVAKIA""","""historical_entity""","""HIST_CZECHOSLOVAKIA""","""Czechoslovakia""",true,false,4,1989,1992,4,1989,1992,null,"""No additive fatality metric used here; UCDP OV contains cumulative analysis-unit-year fields.""","""No unique successor state can be selected without arbitrary allocation between Czechia and Slovakia; the source label is…","""Low UCDP impact, but retained for methodological consistency.""","""high"""
"""ucdp_organized_violence_cy""","""German Democratic Republic""","""keep_as_non_iso_analysis_unit""","""historical_entity""",null,"""German Democratic Republic""","""HIST_GDR""","""historical_entity""","""HIST_GDR""","""German Democratic Republic""",true,false,2,1989,1990,2,1989,1990,null,"""No additive fatality metric used here; UCDP OV contains cumulative analysis-unit-year fields.""","""East Germany is not assigned to contemporary Germany. It is retained as a non-ISO historical analysis unit so the violen…","""Very low UCDP impact, but retained for methodological consistency.""","""high"""
"""ucdp_organized_violence_cy""","""Serbia (Yugoslavia)""","""keep_as_non_iso_analysis_unit""","""historical_composite_entity""",null,"""Serbia (Yugoslavia)""","""HIST_SERBIA_YUGOSLAVIA""","""historical_composite_entity""","""HIST_SERBIA_YUGOSLAVIA""","""Serbia (Yugoslavia)""",true,false,37,1989,2025,37,1989,2025,null,"""No additive fatality metric used here; UCDP OV contains cumulative analysis-unit-year fields.""","""Yugoslav-war and Serbia/Montenegro-era observations are retained as a non-ISO historical composite analysis unit rather …","""High UCDP impact; retention as a non-ISO unit avoids both data loss and arbitrary remapping.""","""high"""
"""ucdp_organized_violence_cy""","""Yemen (North Yemen)""","""keep_as_non_iso_analysis_unit""","""historical_entity""",null,"""Yemen (North Yemen)""","""HIST_NORTH_YEMEN""","""historical_entity""","""HIST_NORTH_YEMEN""","""Yemen (North Yemen)""",true,false,37,1989,2025,37,1989,2025,null,"""No additive fatality metric used here; UCDP OV contains cumulative analysis-unit-year fields.""","""North Yemen before unification is retained as a non-ISO historic

### What this output tells us

The impact report turns the country-crosswalk arbitration into an auditable quantitative object. For mapped cases, it confirms the amount of data preserved. For excluded or outside-ISO3 cases, it shows the volume deliberately left out of the analysis-unit panel with ISO3 as an optional attribute. Notebook 03 should mention these exclusions whenever interpreting analysis-unit-level results for historically complex regions.


### Consequence for the next step

The crosswalk arbitration is no longer an open blocker for Notebook 03. The next transformations can use the documented V1 decisions directly.

The analysis-unit-year panel remains a standard ISO3-oriented panel for mapped countries and validated special statistical units. Historical or composite entities that were explicitly resolved outside ISO3 are not force-mapped into contemporary states.

This is the key handoff point: Notebook 03 can start substantive country-year analysis from a cleaner panel, but it should preserve the methodological caveats documented here. In particular, Russia/Soviet Union, Serbia/Yugoslavia, Yemen/North Yemen and Kosovo should not be described casually as ordinary ISO3 mappings.


<a id="complete-backbone"></a>

<div class="cl-responsive cl-section-block" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 7</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">Complete country-year backbone</h2>
  <p style='margin:8px 0 0 0;color:#D9E2EC;line-height:1.45;'>The panel starts from the analytical grid, not from conflict events.</p>
</div>


The backbone is the structural core of the panel. It is built independently from any one conflict source.

This is essential because a country-year missing from a conflict dataset can mean different things:

| Situation | Interpretation |
|---|---|
| Country-year exists in the backbone, source covers the year, no source row joined | Potential observed zero. |
| Country-year exists in the backbone, source does not cover the year | Missing because out of source coverage. |
| Country-year exists in the backbone, raw country was unresolved | Crosswalk issue. |
| Country-year absent from the backbone | Structural failure of panel construction. |

By starting from a complete grid, the notebook avoids building a panel that only contains violent country-years.


### Why this cell exists — build the panel backbone independently from conflict rows

The backbone must not be built from UCDP or ACLED observations only. If we did that, country-years without observed violence would disappear, and we could no longer distinguish observed zeros from missing coverage.

This cell builds the complete `country_iso3 × year` grid from the country universe and staging years. It creates the stable panel key before any source is joined.

In [14]:
# analysis-unit backbone
vdem_iso = (
    vdem
    .select([pl.col('country_text_id').alias('country_iso3'), pl.col('country_name').alias('vdem_country_name')])
    .drop_nulls('country_iso3')
    .unique()
    .with_columns(pl.col('country_iso3').cast(pl.Utf8).str.to_uppercase().str.strip_chars())
    .filter(pl.col('country_iso3').is_in(list(VALID_ISO3)))
)
wdi_iso = (
    wdi
    .select([pl.col('iso3').alias('country_iso3'), pl.col('country').alias('wdi_country_name')])
    .drop_nulls('country_iso3')
    .unique()
    .with_columns(pl.col('country_iso3').cast(pl.Utf8).str.to_uppercase().str.strip_chars())
    .filter(pl.col('country_iso3').is_in(list(VALID_ISO3)))
)
conflict_iso_auto = (
    country_crosswalk
    .filter(pl.col('country_iso3').is_not_null())
    .select([pl.col('country_iso3'), pl.col('country_name').alias('conflict_country_name')])
    .unique()
)
conflict_iso_arbitrated = (
    country_crosswalk_arbitration_v1
    .filter(pl.col('target_iso3').is_not_null())
    .select([
        pl.col('target_iso3').alias('country_iso3'),
        pl.col('target_country_name').alias('conflict_country_name'),
    ])
    .unique()
)
conflict_iso = pl.concat([conflict_iso_auto, conflict_iso_arbitrated], how='vertical').unique()

# Name lookup for ISO units.
name_lookup = {}
for frame, col in [(wdi_iso, 'wdi_country_name'), (vdem_iso, 'vdem_country_name'), (conflict_iso, 'conflict_country_name')]:
    for row in frame.drop_nulls('country_iso3').iter_rows(named=True):
        if row.get(col) is not None and not name_lookup.get(row['country_iso3']):
            name_lookup[row['country_iso3']] = row[col]

def best_country_name(iso3):
    return iso3_to_name(iso3) or name_lookup.get(iso3)

iso_unit_universe = (
    pl.concat([vdem_iso.select('country_iso3'), wdi_iso.select('country_iso3'), conflict_iso.select('country_iso3')])
    .unique()
    .sort('country_iso3')
    .with_columns([
        pl.col('country_iso3').alias('analysis_unit_id'),
        pl.col('country_iso3').map_elements(best_country_name, return_dtype=pl.Utf8).alias('analysis_unit_label'),
        pl.lit('iso3_country').alias('analysis_unit_type'),
        pl.col('country_iso3').map_elements(best_country_name, return_dtype=pl.Utf8).alias('country_name'),
        pl.lit(True).alias('is_iso3_country'),
        pl.lit(False).alias('is_non_iso_analysis_unit'),
    ])
    .select(['analysis_unit_id', 'analysis_unit_label', 'analysis_unit_type', 'country_iso3', 'country_name', 'is_iso3_country', 'is_non_iso_analysis_unit'])
)

non_iso_unit_universe = (
    country_crosswalk_arbitration_v1
    .filter((pl.col('proposed_action') == 'keep_as_non_iso_analysis_unit') & pl.col('geo_unit_id').is_not_null())
    .select([
        pl.col('geo_unit_id').alias('analysis_unit_id'),
        pl.when(pl.col('target_country_name').is_not_null()).then(pl.col('target_country_name')).otherwise(pl.col('raw_country')).alias('analysis_unit_label'),
        pl.col('geo_unit_type').fill_null('non_iso_analysis_unit').alias('analysis_unit_type'),
        pl.lit(None).cast(pl.Utf8).alias('country_iso3'),
        pl.lit(None).cast(pl.Utf8).alias('country_name'),
        pl.lit(False).alias('is_iso3_country'),
        pl.lit(True).alias('is_non_iso_analysis_unit'),
    ])
    .unique(subset=['analysis_unit_id'])
    .sort('analysis_unit_id')
)

analysis_unit_universe = pl.concat([iso_unit_universe, non_iso_unit_universe], how='vertical').unique(subset=['analysis_unit_id']).sort('analysis_unit_id')

panel_years = pl.DataFrame({'year': list(range(STAGING_YEAR_START, STAGING_YEAR_END + 1))})
backbone = (
    analysis_unit_universe
    .join(panel_years, how='cross')
    .with_columns([
        (pl.col('analysis_unit_id') + '_' + pl.col('year').cast(pl.Utf8)).alias('panel_key'),
        pl.col('year').is_between(RECOMMENDED_YEAR_START, RECOMMENDED_YEAR_END).alias('panel_year_in_recommended_window'),
    ])
    .select([
        'analysis_unit_id', 'analysis_unit_label', 'analysis_unit_type',
        'country_iso3', 'country_name', 'is_iso3_country', 'is_non_iso_analysis_unit',
        'year', 'panel_key', 'panel_year_in_recommended_window'
    ])
)

backbone_summary = pl.DataFrame([
    {"metric": "analysis_units_in_backbone", "value": int(backbone.select(pl.col("analysis_unit_id").n_unique()).item())},
    {"metric": "iso3_units_in_backbone", "value": int(backbone.select(pl.col("is_iso3_country").sum()).item() / backbone.select(pl.col("year").n_unique()).item())},
    {"metric": "non_iso_units_in_backbone", "value": int(backbone.select(pl.col("is_non_iso_analysis_unit").sum()).item() / backbone.select(pl.col("year").n_unique()).item())},
    {"metric": "years_in_backbone", "value": int(backbone.select(pl.col("year").n_unique()).item())},
    {"metric": "rows_in_backbone", "value": backbone.height},
    {"metric": "year_start", "value": int(backbone.select(pl.col("year").min()).item())},
    {"metric": "year_end", "value": int(backbone.select(pl.col("year").max()).item())},
])

backbone_summary

metric,value
str,i64
"""analysis_units_in_backbone""",251
"""iso3_units_in_backbone""",246
"""non_iso_units_in_backbone""",5
"""years_in_backbone""",38
"""rows_in_backbone""",9538
"""year_start""",1989
"""year_end""",2026


### What this output tells us

The backbone output should show a complete grid with one row per `analysis_unit_id + year`. This confirms that the panel exists independently of conflict observations.

From this point onward, a missing value in a source column does not mean the country-year is missing from the panel. It means the joined source did not provide a usable value for that country-year, and must be interpreted with coverage/source flags.

<a id="source-transformations"></a>

<div class="cl-responsive cl-section-block" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 8</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">Source-specific country-year transformations</h2>
  <p style='margin:8px 0 0 0;color:#D9E2EC;line-height:1.45;'>Each source is reduced to the panel grain before joining.</p>
</div>


The following transformations all serve the same purpose: produce one unique row per `analysis_unit_id + year` for each source-specific table.

A source can only be safely joined to the backbone after it passes this grain check. The join report later verifies that no transformed source creates duplicated country-year keys.


### Why this cell exists — define transformation and join-quality helpers

Every source must be reduced to the same grain: one row per `analysis_unit_id + year`. We also need repeated checks for duplicate keys and join coverage.

This cell defines helper functions used by the source-specific transformations and the join report. It prepares the quality gates but does not yet transform a source.

In [15]:
def apply_v1_crosswalk_arbitration(df: pl.DataFrame, source: str, year_col: str = "year") -> pl.DataFrame:
    work = df.with_columns([
        pl.lit(None).cast(pl.Utf8).alias("crosswalk_action_v1"),
        pl.lit(None).cast(pl.Utf8).alias("crosswalk_category_v1"),
        pl.lit(None).cast(pl.Utf8).alias("crosswalk_geo_unit_id_v1"),
        pl.lit(None).cast(pl.Utf8).alias("crosswalk_geo_unit_type_v1"),
        pl.col("country_iso3").alias("analysis_unit_id"),
        pl.col("country_name").alias("analysis_unit_label"),
        pl.when(pl.col("country_iso3").is_not_null()).then(pl.lit("iso3_country")).otherwise(None).alias("analysis_unit_type"),
    ])
    rules = country_crosswalk_arbitration_v1.filter(pl.col("source") == source).iter_rows(named=True)
    for rule in rules:
        start = rule.get("decision_year_min")
        end = rule.get("decision_year_max")
        year_expr = pl.lit(True)
        if start is not None:
            year_expr = year_expr & (pl.col(year_col).cast(pl.Int64, strict=False) >= int(start))
        if end is not None:
            year_expr = year_expr & (pl.col(year_col).cast(pl.Int64, strict=False) <= int(end))
        applies = (pl.col("raw_country") == rule["raw_country"]) & year_expr
        target_iso3 = rule.get("target_iso3")
        target_country_name = rule.get("target_country_name")
        geo_unit_id = rule.get("geo_unit_id")
        geo_unit_type = rule.get("geo_unit_type")
        proposed_action = rule.get("proposed_action")
        target_is_not_null = target_iso3 is not None
        keep_non_iso = (proposed_action == "keep_as_non_iso_analysis_unit") and (geo_unit_id is not None) and (target_iso3 is None)
        non_iso_label = target_country_name or rule.get("raw_country")
        work = work.with_columns([
            pl.when(applies).then(pl.lit(proposed_action)).otherwise(pl.col("crosswalk_action_v1")).alias("crosswalk_action_v1"),
            pl.when(applies).then(pl.lit(rule.get("category"))).otherwise(pl.col("crosswalk_category_v1")).alias("crosswalk_category_v1"),
            pl.when(applies).then(pl.lit(geo_unit_id)).otherwise(pl.col("crosswalk_geo_unit_id_v1")).alias("crosswalk_geo_unit_id_v1"),
            pl.when(applies).then(pl.lit(geo_unit_type)).otherwise(pl.col("crosswalk_geo_unit_type_v1")).alias("crosswalk_geo_unit_type_v1"),

            # ISO3 mapping remains exactly the upstream arbitration decision.
            pl.when(applies & pl.lit(target_is_not_null)).then(pl.lit(target_iso3)).otherwise(pl.col("country_iso3")).alias("country_iso3"),
            pl.when(applies & pl.lit(target_is_not_null)).then(pl.lit(target_country_name)).otherwise(pl.col("country_name")).alias("country_name"),

            # Analytical unit: ISO3 countries use ISO3; non-ISO historical/composite cases use the validated geo_unit_id.
            pl.when(applies & pl.lit(target_is_not_null)).then(pl.lit(target_iso3))
              .when(applies & pl.lit(keep_non_iso)).then(pl.lit(geo_unit_id))
              .otherwise(pl.col("analysis_unit_id")).alias("analysis_unit_id"),
            pl.when(applies & pl.lit(target_is_not_null)).then(pl.lit(target_country_name))
              .when(applies & pl.lit(keep_non_iso)).then(pl.lit(non_iso_label))
              .otherwise(pl.col("analysis_unit_label")).alias("analysis_unit_label"),
            pl.when(applies & pl.lit(target_is_not_null)).then(pl.lit("iso3_country"))
              .when(applies & pl.lit(keep_non_iso)).then(pl.lit(geo_unit_type or "non_iso_analysis_unit"))
              .otherwise(pl.col("analysis_unit_type")).alias("analysis_unit_type"),

            pl.when(applies & (pl.lit(target_is_not_null) | pl.lit(keep_non_iso))).then(pl.lit("matched"))
              .otherwise(pl.col("match_status")).alias("match_status"),
            pl.when(applies).then(pl.lit("v1_crosswalk_arbitration")).otherwise(pl.col("match_method")).alias("match_method"),
        ])
    return work.with_columns([
        (pl.col("country_iso3").is_not_null() & (pl.col("analysis_unit_id") == pl.col("country_iso3"))).alias("is_iso3_country"),
        (pl.col("analysis_unit_id").is_not_null() & pl.col("country_iso3").is_null()).alias("is_non_iso_analysis_unit"),
    ])


def attach_crosswalk(df: pl.DataFrame, source: str, raw_col: str, year_col: str = "year") -> pl.DataFrame:
    xw = country_crosswalk.filter(pl.col('source') == source).select(['raw_country', 'country_iso3', 'country_name', 'match_status', 'match_method'])
    work = df.with_columns(pl.col(raw_col).cast(pl.Utf8).str.strip_chars().alias('raw_country')).join(xw, on='raw_country', how='left')
    return apply_v1_crosswalk_arbitration(work, source, year_col=year_col)


def duplicate_key_count(df: pl.DataFrame) -> int | None:
    if not {'analysis_unit_id', 'year'}.issubset(set(df.columns)):
        return None
    return int(df.select(pl.struct(['analysis_unit_id', 'year']).is_duplicated().sum()).item())


def source_join_stats(source, raw_df, transformed_df, variables_added, year_col='year'):
    xw = country_crosswalk_resolution.filter(pl.col('source') == source)
    y_min, y_max = min_max_df(raw_df, year_col) if year_col in raw_df.columns else (None, None)
    mapped_statuses = {
        'auto_matched',
        'mapped_by_arbitration',
        'partially_mapped_and_kept_as_non_iso_unit_by_arbitration',
        'kept_as_non_iso_analysis_unit_by_arbitration',
    }
    return {
        'source': source,
        'raw_rows': raw_df.height,
        'raw_country_values': int(xw.select(pl.col('raw_country').n_unique()).item()),
        'matched_or_kept_raw_country_values': int(xw.filter(pl.col('resolution_status').is_in(list(mapped_statuses))).select(pl.col('raw_country').n_unique()).item()),
        'excluded_raw_country_values': int(xw.filter(pl.col('resolution_status') == 'excluded_by_arbitration').select(pl.col('raw_country').n_unique()).item()),
        'unresolved_raw_country_values': int(xw.filter(pl.col('resolution_status') == 'unresolved_after_arbitration').select(pl.col('raw_country').n_unique()).item()),
        'year_min': y_min,
        'year_max': y_max,
        'transformed_rows': transformed_df.height,
        'transformed_analysis_unit_values': int(transformed_df.select(pl.col('analysis_unit_id').n_unique()).item()) if 'analysis_unit_id' in transformed_df.columns else None,
        'duplicate_analysis_unit_year_keys_after_transform': duplicate_key_count(transformed_df),
        'variables_added': ', '.join(variables_added),
    }

### How to interpret this cell

This helper cell normally produces no substantive output. Its value is that all later source transformations use the same duplicate-key and join-report logic.

That consistency matters because a single duplicated country-year key in any source would invalidate the panel grain before joining.

### UCDP Organized Violence CY — primary aggregate conflict source

UCDP Organized Violence CY already uses the analysis-unit-year grain. The transformation therefore focuses on conservative country matching, variable renaming and selection of core aggregate conflict metrics.

The staging panel keeps detailed UCDP OV components, but the analysis-ready panel later retains only selected aggregate variables to avoid overloading the first descriptive analysis.


### Why this cell exists — transform UCDP Organized Violence CY

UCDP Organized Violence CY is already country-year and acts as the primary aggregate conflict source. It still needs country harmonisation, variable selection and key validation before it can be joined.

This cell keeps UCDP OV detail in staging and prepares the metrics that will later be curated for analysis-ready use.

In [16]:
ucdp_ov_mapped = attach_crosswalk(ucdp_ov, 'ucdp_organized_violence_cy', 'country').filter(pl.col('analysis_unit_id').is_not_null())
rename_ov = {'country':'ucdp_ov_raw_country','region':'ucdp_ov_region','sb_exist':'ucdp_ov_sb_exist','sb_dyad_count':'ucdp_ov_sb_dyad_count','sb_total_deaths_best':'ucdp_ov_sb_total_deaths_best','sb_deaths_civilians':'ucdp_ov_sb_deaths_civilians','ns_exist':'ucdp_ov_ns_exist','ns_dyad_count':'ucdp_ov_ns_dyad_count','ns_total_deaths_best':'ucdp_ov_ns_total_deaths_best','ns_deaths_civilians':'ucdp_ov_ns_deaths_civilians','os_exist':'ucdp_ov_os_exist','os_dyad_count':'ucdp_ov_os_dyad_count','os_total_deaths_best':'ucdp_ov_os_total_deaths_best','cumulative_total_deaths_civilians_in_orgvio':'ucdp_ov_civilian_deaths_best','cumulative_total_deaths_in_orgvio_best':'ucdp_ov_total_deaths_best'}
keep = ['analysis_unit_id','year'] + [c for c in rename_ov if c in ucdp_ov_mapped.columns]
ucdp_ov_panel = ucdp_ov_mapped.select(keep).rename(rename_ov).with_columns(pl.col('year').cast(pl.Int64, strict=False))
num_ov = [c for c in ucdp_ov_panel.columns if c.startswith('ucdp_ov_') and c not in ['ucdp_ov_raw_country','ucdp_ov_region']]
ucdp_ov_panel = ucdp_ov_panel.with_columns([pl.col(c).cast(pl.Float64, strict=False) for c in num_ov])
ucdp_ov_panel = ucdp_ov_panel.with_columns(
    pl.max_horizontal([pl.col('ucdp_ov_sb_exist').fill_null(0),pl.col('ucdp_ov_ns_exist').fill_null(0),pl.col('ucdp_ov_os_exist').fill_null(0)]).cast(pl.Int64).alias('ucdp_ov_any_organized_violence'),
    pl.lit(True).alias('ucdp_ov_has_source_row')
)
ucdp_ov_panel = ucdp_ov_panel.group_by(['analysis_unit_id','year']).agg([
    pl.col('ucdp_ov_raw_country').first(), pl.col('ucdp_ov_region').first(),
    pl.col('ucdp_ov_sb_exist').max(), pl.col('ucdp_ov_sb_dyad_count').sum(), pl.col('ucdp_ov_sb_total_deaths_best').sum(), pl.col('ucdp_ov_sb_deaths_civilians').sum(),
    pl.col('ucdp_ov_ns_exist').max(), pl.col('ucdp_ov_ns_dyad_count').sum(), pl.col('ucdp_ov_ns_total_deaths_best').sum(), pl.col('ucdp_ov_ns_deaths_civilians').sum(),
    pl.col('ucdp_ov_os_exist').max(), pl.col('ucdp_ov_os_dyad_count').sum(), pl.col('ucdp_ov_os_total_deaths_best').sum(),
    pl.col('ucdp_ov_civilian_deaths_best').sum(), pl.col('ucdp_ov_total_deaths_best').sum(),
    pl.col('ucdp_ov_any_organized_violence').max(), pl.col('ucdp_ov_has_source_row').max()
]).sort(['analysis_unit_id','year'])
ucdp_ov_variables = [c for c in ucdp_ov_panel.columns if c.startswith('ucdp_ov_')]

ucdp_ov_panel.head()

analysis_unit_id,year,ucdp_ov_raw_country,ucdp_ov_region,ucdp_ov_sb_exist,ucdp_ov_sb_dyad_count,ucdp_ov_sb_total_deaths_best,ucdp_ov_sb_deaths_civilians,ucdp_ov_ns_exist,ucdp_ov_ns_dyad_count,ucdp_ov_ns_total_deaths_best,ucdp_ov_ns_deaths_civilians,ucdp_ov_os_exist,ucdp_ov_os_dyad_count,ucdp_ov_os_total_deaths_best,ucdp_ov_civilian_deaths_best,ucdp_ov_total_deaths_best,ucdp_ov_any_organized_violence,ucdp_ov_has_source_row
str,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,bool
"""AFG""",1989,"""Afghanistan""","""Asia""",1.0,5.0,5174.0,303.0,1.0,2.0,237.0,0.0,0.0,0.0,0.0,303.0,5411.0,1,true
"""AFG""",1990,"""Afghanistan""","""Asia""",1.0,5.0,1478.0,101.0,1.0,1.0,36.0,0.0,0.0,0.0,0.0,101.0,1514.0,1,true
"""AFG""",1991,"""Afghanistan""","""Asia""",1.0,4.0,3302.0,38.0,1.0,2.0,251.0,11.0,0.0,0.0,0.0,49.0,3553.0,1,true
"""AFG""",1992,"""Afghanistan""","""Asia""",1.0,4.0,4287.0,1687.0,1.0,1.0,90.0,0.0,1.0,1.0,8.0,1695.0,4385.0,1,true
"""AFG""",1993,"""Afghanistan""","""Asia""",1.0,4.0,4071.0,611.0,1.0,1.0,8.0,0.0,1.0,1.0,26.0,637.0,4105.0,1,true


### What this output tells us

The UCDP OV panel should now be at analysis-unit-year grain with no duplicated `analysis_unit_id + year` keys. It contributes both detailed staging columns and selected core metrics used later for analysis-ready zero-filled variables.

If duplicate keys appeared here, the source could not be safely joined until the aggregation or country-matching logic was corrected.

### UCDP GED — event-level source aggregated to country-year

UCDP GED starts at the event level. This notebook aggregates it to country-year to recover event counts, deaths and violence-type counts.

This source is important because it preserves an event-derived view even though the first analytical track is country-year. It also keeps the project aligned with the later event-level analysis track.


### Why this cell exists — aggregate UCDP GED from events to country-year

UCDP GED starts from event-level records, while the panel grain is country-year. To make GED comparable with the backbone, we aggregate events by `analysis_unit_id + year`.

This cell produces event counts, death estimates, civilian deaths and counts by violence type. It keeps the event-derived nature of GED visible while enforcing the panel grain.

In [17]:
ucdp_ged_mapped = attach_crosswalk(ucdp_ged, 'ucdp_ged_events', 'country').filter(pl.col('analysis_unit_id').is_not_null()).with_columns(pl.col('year').cast(pl.Int64, strict=False))
for col in ['best','high','low','deaths_civilians','deaths_a','deaths_b','deaths_unknown','type_of_violence']:
    ucdp_ged_mapped = ucdp_ged_mapped.with_columns(pl.col(col).cast(pl.Float64 if col!='type_of_violence' else pl.Int64, strict=False))
ucdp_ged_mapped = ucdp_ged_mapped.with_columns([
    (pl.col('type_of_violence')==1).cast(pl.Int64).alias('is_state_based_event'),
    (pl.col('type_of_violence')==2).cast(pl.Int64).alias('is_non_state_event'),
    (pl.col('type_of_violence')==3).cast(pl.Int64).alias('is_one_sided_event')
])
ucdp_ged_panel = ucdp_ged_mapped.group_by(['analysis_unit_id','year']).agg([
    pl.len().alias('ucdp_ged_events'),
    pl.col('best').sum().alias('ucdp_ged_best_deaths'),
    pl.col('high').sum().alias('ucdp_ged_high_deaths'),
    pl.col('low').sum().alias('ucdp_ged_low_deaths'),
    pl.col('deaths_civilians').sum().alias('ucdp_ged_civilian_deaths'),
    pl.col('is_state_based_event').sum().alias('ucdp_ged_state_based_events'),
    pl.col('is_non_state_event').sum().alias('ucdp_ged_non_state_events'),
    pl.col('is_one_sided_event').sum().alias('ucdp_ged_one_sided_events'),
    pl.col('conflict_new_id').n_unique().alias('ucdp_ged_conflicts'),
    pl.col('dyad_new_id').n_unique().alias('ucdp_ged_dyads')
]).with_columns(pl.lit(True).alias('ucdp_ged_has_source_row')).sort(['analysis_unit_id','year'])
ucdp_ged_variables = [c for c in ucdp_ged_panel.columns if c.startswith('ucdp_ged_')]

ucdp_ged_panel.head()

analysis_unit_id,year,ucdp_ged_events,ucdp_ged_best_deaths,ucdp_ged_high_deaths,ucdp_ged_low_deaths,ucdp_ged_civilian_deaths,ucdp_ged_state_based_events,ucdp_ged_non_state_events,ucdp_ged_one_sided_events,ucdp_ged_conflicts,ucdp_ged_dyads,ucdp_ged_has_source_row
str,i64,u32,f64,f64,f64,f64,i64,i64,i64,u32,u32,bool
"""AFG""",1989,145,5411.0,17642.0,2141.0,303.0,137,8,0,3,7,true
"""AFG""",1990,45,1514.0,2311.0,1359.0,101.0,39,6,0,2,6,true
"""AFG""",1991,64,3553.0,4053.0,3407.0,49.0,60,4,0,3,6,true
"""AFG""",1992,76,4385.0,5865.0,4368.0,1695.0,69,3,4,3,6,true
"""AFG""",1993,115,4105.0,8433.0,4097.0,637.0,105,1,9,4,6,true


### What this output tells us

GED has been collapsed from event records to country-year indicators. The resulting table should contain event counts, death estimates, civilian deaths and violence-type counts by country-year.

This transformation makes GED useful for country-year diagnostics while preserving the fact that GED started as event-level data. Event-level analysis remains a separate track for later notebooks.

### UCDP One-sided — civilian violence source

UCDP One-sided is transformed separately because it targets a key substantive theme of ConflictLens: violence against civilians.

Some records list several countries in the location field. The notebook therefore creates an allocated fatality measure to avoid mechanically counting the full fatality estimate once for each listed country.


### Why this cell exists — transform UCDP One-sided Violence with allocation logic

One-sided violence is central for the civilian-exposure axis, but source records can contain multi-country locations. A analysis-unit-year panel cannot use multi-country totals naively without risking double counting.

This cell splits location labels, attaches the crosswalk, allocates fatalities across matched country components, and aggregates the result to country-year.

In [18]:
ucdp_os_work = ucdp_os.with_columns([
    pl.col('location').cast(pl.Utf8).str.split(',').alias('location_components'),
    pl.col('year').cast(pl.Int64, strict=False)
]).with_columns(pl.col('location_components').list.len().alias('ucdp_os_location_count')).explode('location_components').rename({'location_components': 'raw_country'}).with_columns(pl.col('raw_country').cast(pl.Utf8).str.strip_chars())
ucdp_os_work = attach_crosswalk(ucdp_os_work, 'ucdp_one_sided_locations', 'raw_country').filter(pl.col('analysis_unit_id').is_not_null())
ucdp_os_work = ucdp_os_work.with_columns([pl.col(c).cast(pl.Float64, strict=False) for c in ['best_fatality_estimate', 'low_fatality_estimate', 'high_fatality_estimate']] + [pl.col('is_government_actor').cast(pl.Float64, strict=False)])
ucdp_os_work = ucdp_os_work.with_columns([
    (pl.col('best_fatality_estimate') / pl.col('ucdp_os_location_count')).alias('ucdp_os_best_fatalities_allocated'),
    (pl.col('low_fatality_estimate') / pl.col('ucdp_os_location_count')).alias('ucdp_os_low_fatalities_allocated'),
    (pl.col('high_fatality_estimate') / pl.col('ucdp_os_location_count')).alias('ucdp_os_high_fatalities_allocated'),
    (pl.col('ucdp_os_location_count') > 1).alias('ucdp_os_is_multicountry_record'),
])
ucdp_os_panel = ucdp_os_work.group_by(['analysis_unit_id', 'year']).agg([
    pl.len().alias('ucdp_os_records'),
    pl.col('ucdp_os_best_fatalities_allocated').sum(),
    pl.col('ucdp_os_low_fatalities_allocated').sum(),
    pl.col('ucdp_os_high_fatalities_allocated').sum(),
    pl.col('best_fatality_estimate').sum().alias('ucdp_os_best_fatalities_full_multicountry'),
    pl.col('is_government_actor').sum().alias('ucdp_os_government_actor_records'),
    pl.col('ucdp_os_is_multicountry_record').sum().alias('ucdp_os_multicountry_records'),
    pl.when(pl.col('ucdp_os_is_multicountry_record')).then(pl.col('ucdp_os_best_fatalities_allocated')).otherwise(0).sum().alias('ucdp_os_multicountry_best_fatalities_allocated'),
]).with_columns(pl.lit(True).alias('ucdp_os_has_source_row')).sort(['analysis_unit_id', 'year'])
ucdp_os_variables = [c for c in ucdp_os_panel.columns if c.startswith('ucdp_os_')]

ucdp_os_multicountry_allocation_report = (
    ucdp_os_work
    .filter(pl.col('ucdp_os_is_multicountry_record'))
    .group_by(['analysis_unit_id', 'analysis_unit_label', 'analysis_unit_type', 'country_iso3', 'country_name', 'year'])
    .agg([
        pl.len().alias('multicountry_records'),
        pl.col('ucdp_os_best_fatalities_allocated').sum().alias('allocated_best_fatalities'),
        pl.col('best_fatality_estimate').sum().alias('full_best_fatalities_before_allocation'),
    ])
    .sort(['year', 'analysis_unit_id'])
)

ucdp_os_panel.head()

analysis_unit_id,year,ucdp_os_records,ucdp_os_best_fatalities_allocated,ucdp_os_low_fatalities_allocated,ucdp_os_high_fatalities_allocated,ucdp_os_best_fatalities_full_multicountry,ucdp_os_government_actor_records,ucdp_os_multicountry_records,ucdp_os_multicountry_best_fatalities_allocated,ucdp_os_has_source_row
str,i64,u32,f64,f64,f64,f64,f64,u32,f64,bool
"""AFG""",1993,1,26.0,26.0,240.0,26.0,1.0,0,0.0,true
"""AFG""",1995,1,70.5,70.5,1235.5,141.0,1.0,1,70.5,true
"""AFG""",1996,1,42.0,12.0,42.0,42.0,0.0,0,0.0,true
"""AFG""",1997,1,323.0,323.0,323.0,323.0,1.0,0,0.0,true
"""AFG""",1998,1,5801.0,3731.0,7381.0,5801.0,1.0,0,0.0,true


### What this output tells us

The UCDP one-sided output should show allocated country-year measures rather than raw multi-country records. The `best` allocated fatalities become the main analysis-ready candidate, while low/high bounds and multicountry diagnostics stay available in staging.

This preserves civilian-violence information without hiding allocation assumptions.

### ACLED country-year summaries — complementary benchmark

ACLED is not used as the primary conflict definition in this panel. It is integrated as a benchmark layer that can later be compared with UCDP-derived measures.

The notebook uses the available country-year summary tables rather than raw ACLED events. This is sufficient for a first benchmark but should be revisited if advanced ACLED metrics become part of Phase 2BIS.


### Why this cell exists — prepare ACLED benchmark metrics

ACLED is not the primary conflict source in this panel. It is a complementary benchmark with different definitions and a partial current-year horizon.

This cell converts ACLED country-year summary workbooks into comparable benchmark columns: political violence events, reported fatalities, civilian-targeting events and reported civilian fatalities.

In [19]:
def acled_to_panel(df, source, value_col, output_col):
    work = df.select([
        pl.col('COUNTRY').alias('raw_country'),
        pl.col('YEAR').cast(pl.Int64, strict=False).alias('year'),
        pl.col(value_col).cast(pl.Float64, strict=False).alias(output_col)
    ]).with_columns(pl.col('raw_country').cast(pl.Utf8).str.strip_chars())
    work = attach_crosswalk(work, source, 'raw_country').filter(pl.col('analysis_unit_id').is_not_null())
    return work.group_by(['analysis_unit_id', 'year']).agg(pl.col(output_col).sum().alias(output_col)).sort(['analysis_unit_id', 'year'])

acled_events_panel = acled_to_panel(acled_events, 'acled_political_violence_events', 'EVENTS', 'acled_political_violence_events')
acled_fatalities_panel = acled_to_panel(acled_fatalities, 'acled_reported_fatalities', 'FATALITIES', 'acled_reported_fatalities')
acled_civ_events_panel = acled_to_panel(acled_civ_events, 'acled_civilian_targeting_events', 'EVENTS', 'acled_civilian_targeting_events')
acled_civ_fatalities_panel = acled_to_panel(acled_civ_fatalities, 'acled_reported_civilian_fatalities', 'FATALITIES', 'acled_reported_civilian_fatalities')
acled_panel = acled_events_panel.join(acled_fatalities_panel, on=['analysis_unit_id', 'year'], how='full', coalesce=True).join(acled_civ_events_panel, on=['analysis_unit_id', 'year'], how='full', coalesce=True).join(acled_civ_fatalities_panel, on=['analysis_unit_id', 'year'], how='full', coalesce=True)
acled_year_max = int(acled_panel.select(pl.col('year').max()).item())
acled_panel = acled_panel.with_columns(pl.lit(True).alias('acled_has_any_source_row'), (pl.col('year') == acled_year_max).alias('acled_year_is_partial'))
acled_variables = [c for c in acled_panel.columns if c.startswith('acled_')]

acled_panel.head()

analysis_unit_id,year,acled_political_violence_events,acled_reported_fatalities,acled_civilian_targeting_events,acled_reported_civilian_fatalities,acled_has_any_source_row,acled_year_is_partial
str,i64,f64,f64,f64,f64,bool,bool
"""ABW""",2018,0.0,0.0,0.0,0.0,true,false
"""ABW""",2019,0.0,0.0,0.0,0.0,true,false
"""ABW""",2020,0.0,0.0,0.0,0.0,true,false
"""ABW""",2021,0.0,0.0,0.0,0.0,true,false
"""ABW""",2022,null,0.0,null,null,true,false


### What this output tells us

ACLED contributes benchmark measures, not the primary conflict definition. Its variables allow later comparison with UCDP-based patterns, especially for political violence and civilian targeting.

The partial-year flag is important: ACLED 2026 values should not be compared naively with complete historical years.

### V-Dem and WDI — minimal context layer

The context layer is intentionally small. It gives Notebook 03 enough variables for first descriptive analysis without turning Notebook 02 into a broad enrichment pipeline.

V-Dem contributes selected institutional indicators. WDI contributes selected socioeconomic and demographic indicators. Additional variables can be added later, but only after the minimal panel has been validated.


### Why this cell exists — prepare minimal V-Dem and WDI context variables

The first analysis-unit-year panel should include enough political and socio-economic context for descriptive analysis, without turning Notebook 02 into an enrichment notebook.

This cell keeps a small V1 subset from V-Dem and WDI, validates country/year keys, and defers broader enrichment to Phase 2BIS.

In [20]:
vdem_panel = (
    vdem
    .rename({'country_text_id':'country_iso3','country_name':'vdem_country_name','v2x_polyarchy':'vdem_electoral_democracy','v2x_libdem':'vdem_liberal_democracy','v2x_delibdem':'vdem_deliberative_democracy','v2x_egaldem':'vdem_egalitarian_democracy','v2x_rule':'vdem_rule_of_law','v2x_corr':'vdem_political_corruption'})
    .with_columns([pl.col('country_iso3').cast(pl.Utf8).str.to_uppercase().str.strip_chars(), pl.col('year').cast(pl.Int64, strict=False)])
    .filter(pl.col('country_iso3').is_in(list(VALID_ISO3)))
    .with_columns(pl.col('country_iso3').alias('analysis_unit_id'))
    .select(['analysis_unit_id','year','vdem_country_name','vdem_electoral_democracy','vdem_liberal_democracy','vdem_deliberative_democracy','vdem_egalitarian_democracy','vdem_rule_of_law','vdem_political_corruption'])
    .unique(subset=['analysis_unit_id','year'])
    .with_columns(pl.lit(True).alias('vdem_has_source_row'))
)
vdem_variables = [c for c in vdem_panel.columns if c.startswith('vdem_')]

wdi_panel = (
    wdi
    .rename({'iso3':'country_iso3','country':'wdi_country_name','gdp_per_capita_constant_2015_usd':'wdi_gdp_per_capita_constant_2015_usd','gini_index':'wdi_gini_index','internet_users_pct_population':'wdi_internet_users_pct_population','population_total':'wdi_population_total','secondary_school_enrollment_gross_pct':'wdi_secondary_school_enrollment_gross_pct','under5_mortality_rate':'wdi_under5_mortality_rate','urban_population_pct_total':'wdi_urban_population_pct_total'})
    .with_columns([pl.col('country_iso3').cast(pl.Utf8).str.to_uppercase().str.strip_chars(), pl.col('year').cast(pl.Int64, strict=False)])
    .filter(pl.col('country_iso3').is_in(list(VALID_ISO3)))
    .with_columns(pl.col('country_iso3').alias('analysis_unit_id'))
    .select(['analysis_unit_id','year','wdi_country_name','wdi_gdp_per_capita_constant_2015_usd','wdi_gini_index','wdi_internet_users_pct_population','wdi_population_total','wdi_secondary_school_enrollment_gross_pct','wdi_under5_mortality_rate','wdi_urban_population_pct_total'])
    .unique(subset=['analysis_unit_id','year'])
    .with_columns(pl.lit(True).alias('wdi_has_source_row'))
)
wdi_variables = [c for c in wdi_panel.columns if c.startswith('wdi_')]

pl.DataFrame([
    {"source": "vdem_selected_context", "rows": vdem_panel.height, "analysis_units": int(vdem_panel.select(pl.col("analysis_unit_id").n_unique()).item()), "year_min": int(vdem_panel.select(pl.col("year").min()).item()), "year_max": int(vdem_panel.select(pl.col("year").max()).item())},
    {"source": "wdi_selected_context", "rows": wdi_panel.height, "analysis_units": int(wdi_panel.select(pl.col("analysis_unit_id").n_unique()).item()), "year_min": int(wdi_panel.select(pl.col("year").min()).item()), "year_max": int(wdi_panel.select(pl.col("year").max()).item())},
])

source,rows,analysis_units,year_min,year_max
str,i64,i64,i64,i64
"""vdem_selected_context""",26052,176,1789,2025
"""wdi_selected_context""",7992,216,1989,2025


### What this output tells us

V-Dem and WDI now provide the minimal context layer for Axis A and future normalisations. The selected variables are intentionally limited: enough to support the first descriptive panel, but not so many that Notebook 02 becomes an enrichment exercise.

Sparse context indicators remain `NA`; they are never zero-filled.

<a id="panel-joins"></a>

<div class="cl-responsive cl-section-block" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 9</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">Panel joins and diagnostics</h2>
  <p style='margin:8px 0 0 0;color:#D9E2EC;line-height:1.45;'>Joining is treated as a quality gate.</p>
</div>


At this point, each source has been transformed to the same grain as the backbone. The join step should therefore be structurally simple: left-join each source onto the complete country-year grid.

The key quality expectation is strict: **no source is allowed to create duplicate `analysis_unit_id + year` keys after transformation**. If a source violates this, the notebook should fail rather than silently duplicate panel rows.


### Why this cell exists — join transformed sources onto the backbone

At this point, every source-specific object should be at the same grain. The panel can now be built by left-joining each source onto the complete backbone.

This cell creates the staging panel. The staging panel is intentionally wide: it keeps source details, diagnostics and flags so that the construction remains auditable.

In [21]:
# joins
staging_panel = backbone
source_frames = [('ucdp_ov',ucdp_ov_panel),('ucdp_ged',ucdp_ged_panel),('ucdp_os',ucdp_os_panel),('acled',acled_panel),('vdem',vdem_panel),('wdi',wdi_panel)]
for label, frame in source_frames:
    dup = duplicate_key_count(frame)
    if dup:
        raise ValueError(f'{label} has duplicated analysis-unit-year keys: {dup}')
    before = staging_panel.height
    staging_panel = staging_panel.join(frame, on=['analysis_unit_id','year'], how='left')
    assert staging_panel.height == before, f'Join with {label} changed row count'

for col in ['ucdp_ov_has_source_row','ucdp_ged_has_source_row','ucdp_os_has_source_row','acled_has_any_source_row','vdem_has_source_row','wdi_has_source_row']:
    staging_panel = staging_panel.with_columns(pl.col(col).fill_null(False).cast(pl.Boolean))

coverage_lookup = {r['source']:{'year_min':r['year_min'],'year_max':r['year_max']} for r in source_coverage.iter_rows(named=True)}
def add_source_coverage_flag(df, prefix, source_name):
    info = coverage_lookup[source_name]
    return df.with_columns(pl.col('year').is_between(int(info['year_min']), int(info['year_max'])).alias(f'{prefix}_in_temporal_coverage'))

staging_panel = add_source_coverage_flag(staging_panel,'ucdp_ov','ucdp_organized_violence_cy')
staging_panel = add_source_coverage_flag(staging_panel,'ucdp_ged','ucdp_ged_events')
staging_panel = add_source_coverage_flag(staging_panel,'ucdp_os','ucdp_one_sided')
staging_panel = add_source_coverage_flag(staging_panel,'vdem','vdem_selected_context')
staging_panel = add_source_coverage_flag(staging_panel,'wdi','wdi_selected_context')
acled_year_min = int(source_coverage.filter(pl.col('family')=='ACLED').select(pl.col('year_min').min()).item())
acled_year_max = int(source_coverage.filter(pl.col('family')=='ACLED').select(pl.col('year_max').max()).item())
staging_panel = staging_panel.with_columns([pl.col('year').is_between(acled_year_min,acled_year_max).alias('acled_in_temporal_coverage'), (pl.col('year')==acled_year_max).alias('acled_year_is_partial')])

# Analysis-unit universe flags make zero-filling safer. A selected conflict metric can be zero-filled only
# when the source covers the year and the analysis unit is represented in that source's harmonised universe.
source_analysis_unit_universe = {
    'ucdp_ov': ucdp_ov_panel.select('analysis_unit_id').unique().to_series().to_list(),
    'ucdp_ged': ucdp_ged_panel.select('analysis_unit_id').unique().to_series().to_list(),
    'ucdp_os': ucdp_os_panel.select('analysis_unit_id').unique().to_series().to_list(),
    'acled': acled_panel.select('analysis_unit_id').unique().to_series().to_list(),
}

def unit_year_bounds(frame: pl.DataFrame, prefix: str) -> pl.DataFrame:
    return (
        frame
        .group_by('analysis_unit_id')
        .agg([
            pl.col('year').min().alias(f'{prefix}_analysis_unit_year_min'),
            pl.col('year').max().alias(f'{prefix}_analysis_unit_year_max'),
        ])
    )

for prefix, frame in [('ucdp_ov', ucdp_ov_panel), ('ucdp_ged', ucdp_ged_panel), ('ucdp_os', ucdp_os_panel), ('acled', acled_panel)]:
    staging_panel = staging_panel.join(unit_year_bounds(frame, prefix), on='analysis_unit_id', how='left')

analysis_core_macro_codes = vdem_iso.select('country_iso3').unique().to_series().to_list()
conflict_analysis_units = sorted(set().union(*[set(v) for v in source_analysis_unit_universe.values()]))
staging_panel = staging_panel.with_columns([
    pl.col('analysis_unit_id').is_in(source_analysis_unit_universe['ucdp_ov']).alias('ucdp_ov_analysis_unit_in_source_universe'),
    pl.col('analysis_unit_id').is_in(source_analysis_unit_universe['ucdp_ged']).alias('ucdp_ged_analysis_unit_in_source_universe'),
    pl.col('analysis_unit_id').is_in(source_analysis_unit_universe['ucdp_os']).alias('ucdp_os_analysis_unit_in_source_universe'),
    pl.col('analysis_unit_id').is_in(source_analysis_unit_universe['acled']).alias('acled_analysis_unit_in_source_universe'),
    (pl.col('is_iso3_country') & pl.col('country_iso3').is_in(analysis_core_macro_codes)).alias('analysis_core_macro_universe'),
    pl.col('analysis_unit_id').is_in(conflict_analysis_units).alias('analysis_conflict_universe'),
])

for prefix in ['ucdp_ov', 'ucdp_ged', 'ucdp_os', 'acled']:
    min_col = f'{prefix}_analysis_unit_year_min'
    max_col = f'{prefix}_analysis_unit_year_max'
    staging_panel = staging_panel.with_columns(
        (
            pl.col('is_iso3_country') |
            ((pl.col('year') >= pl.col(min_col)) & (pl.col('year') <= pl.col(max_col))).fill_null(False)
        ).alias(f'{prefix}_analysis_unit_year_in_source_range')
    )


EXISTENCE_FROM = {
    # Former USSR — independence in 1991
    'ARM': 1991, 'AZE': 1991, 'BLR': 1991, 'EST': 1991, 'GEO': 1991, 'KAZ': 1991,
    'KGZ': 1991, 'LVA': 1991, 'LTU': 1991, 'MDA': 1991, 'TJK': 1991, 'TKM': 1991,
    'UKR': 1991, 'UZB': 1991,
    # Former Yugoslavia
    'HRV': 1991, 'SVN': 1991, 'MKD': 1991, 'BIH': 1992,
    'SRB': 2006, 'MNE': 2006, 'XKX': 2008,
    # Former Czechoslovakia
    'CZE': 1993, 'SVK': 1993,
    # Other independences / unifications
    'NAM': 1990,
    'ERI': 1993,
    'TLS': 2002,
    'SSD': 2011,
    'PLW': 1994,
    'YEM': 1991,
}

staging_panel = staging_panel.with_columns(
    pl.col('analysis_unit_id')
      .replace_strict(EXISTENCE_FROM, default=STAGING_YEAR_START, return_dtype=pl.Int64)
      .alias('unit_exists_from_year')
).with_columns(
    (pl.col('year') >= pl.col('unit_exists_from_year')).alias('unit_exists_in_year')
)

staging_panel = staging_panel.with_columns([
    (pl.col('ucdp_ov_in_temporal_coverage') & pl.col('ucdp_ov_analysis_unit_in_source_universe') & pl.col('ucdp_ov_analysis_unit_year_in_source_range') & pl.col('unit_exists_in_year')).alias('ucdp_ov_can_zero_fill'),
    (pl.col('ucdp_ged_in_temporal_coverage') & pl.col('ucdp_ged_analysis_unit_in_source_universe') & pl.col('ucdp_ged_analysis_unit_year_in_source_range') & pl.col('unit_exists_in_year')).alias('ucdp_ged_can_zero_fill'),
    (pl.col('ucdp_os_in_temporal_coverage') & pl.col('ucdp_os_analysis_unit_in_source_universe') & pl.col('ucdp_os_analysis_unit_year_in_source_range') & pl.col('unit_exists_in_year')).alias('ucdp_os_can_zero_fill'),
    (pl.col('acled_in_temporal_coverage') & pl.col('acled_analysis_unit_in_source_universe') & pl.col('acled_analysis_unit_year_in_source_range') & pl.col('unit_exists_in_year')).alias('acled_can_zero_fill'),
])

staging_panel.shape

(9538, 100)

### What this output tells us

The staging panel is the full audit object. Its width is expected because it contains raw-like source variables, detailed diagnostics, source-row flags and coverage flags.

This output should be used when checking joins, crosswalk decisions or missingness causes. It is not the preferred input for first-pass descriptive charts.

The join report is the main audit trail for how raw sources became panel variables. It tracks raw rows, country matching, temporal coverage, transformed row counts, duplicate-key checks and the variables added by each source.


### Why this cell exists — build the join report quality gate

A multi-source panel is only trustworthy if each join can be audited. We need to know how many rows were read, how many country labels were matched, whether duplicate `analysis_unit_id + year` keys exist, and which variables each source added.

This cell builds `join_report`, the audit trail that explains how raw sources became panel columns.

In [22]:
join_report_rows = [
    source_join_stats('ucdp_organized_violence_cy', ucdp_ov, ucdp_ov_panel, ucdp_ov_variables),
    source_join_stats('ucdp_ged_events', ucdp_ged, ucdp_ged_panel, ucdp_ged_variables),
    source_join_stats('ucdp_one_sided_locations', ucdp_os, ucdp_os_panel, ucdp_os_variables),
    source_join_stats('acled_political_violence_events', acled_events.rename({'YEAR':'year'}), acled_events_panel, ['acled_political_violence_events']),
    source_join_stats('acled_reported_fatalities', acled_fatalities.rename({'YEAR':'year'}), acled_fatalities_panel, ['acled_reported_fatalities']),
    source_join_stats('acled_civilian_targeting_events', acled_civ_events.rename({'YEAR':'year'}), acled_civ_events_panel, ['acled_civilian_targeting_events']),
    source_join_stats('acled_reported_civilian_fatalities', acled_civ_fatalities.rename({'YEAR':'year'}), acled_civ_fatalities_panel, ['acled_reported_civilian_fatalities']),
]
join_report_rows.append({
    'source':'vdem_selected_context',
    'raw_rows':vdem.height,
    'raw_country_values':int(vdem.select(pl.col('country_text_id').n_unique()).item()),
    'matched_or_kept_raw_country_values':int(vdem_panel.select(pl.col('analysis_unit_id').n_unique()).item()),
    'excluded_raw_country_values':0,
    'unresolved_raw_country_values':int(vdem.select(pl.col('country_text_id').n_unique()).item())-int(vdem_panel.select(pl.col('analysis_unit_id').n_unique()).item()),
    'year_min':int(vdem.select(pl.col('year').min()).item()),
    'year_max':int(vdem.select(pl.col('year').max()).item()),
    'transformed_rows':vdem_panel.height,
    'transformed_analysis_unit_values':int(vdem_panel.select(pl.col('analysis_unit_id').n_unique()).item()),
    'duplicate_analysis_unit_year_keys_after_transform':duplicate_key_count(vdem_panel),
    'variables_added':', '.join(vdem_variables)
})
join_report_rows.append({
    'source':'wdi_selected_context',
    'raw_rows':wdi.height,
    'raw_country_values':int(wdi.select(pl.col('iso3').n_unique()).item()),
    'matched_or_kept_raw_country_values':int(wdi_panel.select(pl.col('analysis_unit_id').n_unique()).item()),
    'excluded_raw_country_values':0,
    'unresolved_raw_country_values':int(wdi.select(pl.col('iso3').n_unique()).item())-int(wdi_panel.select(pl.col('analysis_unit_id').n_unique()).item()),
    'year_min':int(wdi.select(pl.col('year').min()).item()),
    'year_max':int(wdi.select(pl.col('year').max()).item()),
    'transformed_rows':wdi_panel.height,
    'transformed_analysis_unit_values':int(wdi_panel.select(pl.col('analysis_unit_id').n_unique()).item()),
    'duplicate_analysis_unit_year_keys_after_transform':duplicate_key_count(wdi_panel),
    'variables_added':', '.join(wdi_variables)
})
join_report = pl.DataFrame(join_report_rows).with_columns([
    pl.lit(staging_panel.height).alias('panel_rows_after_join'),
    pl.lit(int(staging_panel.select(pl.col('analysis_unit_id').n_unique()).item())).alias('panel_analysis_unit_values_after_join'),
    pl.lit(int(staging_panel.select(pl.col('year').min()).item())).alias('panel_year_min'),
    pl.lit(int(staging_panel.select(pl.col('year').max()).item())).alias('panel_year_max')
])

join_report

source,raw_rows,raw_country_values,matched_or_kept_raw_country_values,excluded_raw_country_values,unresolved_raw_country_values,year_min,year_max,transformed_rows,transformed_analysis_unit_values,duplicate_analysis_unit_year_keys_after_transform,variables_added,panel_rows_after_join,panel_analysis_unit_values_after_join,panel_year_min,panel_year_max
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,i32,i32,i32,i32
"""ucdp_organized_violence_cy""",7132,199,199,0,0,1989,2025,7132,200,0,"""ucdp_ov_raw_country, ucdp_ov_region, ucdp_ov_sb_exist, ucdp_ov_sb_dyad_count, ucdp_ov_sb_total_deaths_best, ucdp_ov_sb_d…",9538,251,1989,2026
"""ucdp_ged_events""",417968,126,126,0,0,1989,2025,2132,126,0,"""ucdp_ged_events, ucdp_ged_best_deaths, ucdp_ged_high_deaths, ucdp_ged_low_deaths, ucdp_ged_civilian_deaths, ucdp_ged_sta…",9538,251,1989,2026
"""ucdp_one_sided_locations""",1408,100,100,0,0,1989,2025,1010,100,0,"""ucdp_os_records, ucdp_os_best_fatalities_allocated, ucdp_os_low_fatalities_allocated, ucdp_os_high_fatalities_allocated,…",9538,251,1989,2026
"""acled_political_violence_events""",2754,250,244,6,0,1997,2026,2721,244,0,"""acled_political_violence_events""",9538,251,1989,2026
"""acled_reported_fatalities""",2983,250,244,6,0,1997,2026,2945,244,0,"""acled_reported_fatalities""",9538,251,1989,2026
"""acled_civilian_targeting_events""",2717,250,244,6,0,1997,2026,2691,244,0,"""acled_civilian_targeting_events""",9538,251,1989,2026
"""acled_reported_civilian_fatalities""",2717,250,244,6,0,1997,2026,2691,244,0,"""acled_reported_civilian_fatalities""",9538,251,1989,2026
"""vdem_selected_context""",28092,202,176,0,26,1789,2025,26052,176,0,"""vdem_country_name, vdem_electoral_democracy, vdem_liberal_democracy, vdem_deliberative_democracy, vdem_egalitarian_democ…",9538,251,1989,2026
"""wdi_selected_context""",9620,260,216,0,44,1989,2025,7992,216,0,"""wdi_country_name, wdi_gdp_per_capita_constant_2015_usd, wdi_gini_index, wdi_internet_users_pct_population, wdi_populatio…",9538,251,1989,2026


### What this output tells us

The join report is the main quality gate. The crucial checks are that each transformed source has zero duplicate country-year keys and that unresolved country labels are counted rather than hidden.

If the report shows duplicate keys for a source, the panel should not be used until that source transformation is fixed.

The missingness report is a diagnostic, not a cleaning instruction. High missingness can come from legitimate source coverage limits, sparse contextual indicators, unresolved country matches, or true absence of a source row. The coverage and source-row flags created above are what allow those cases to be interpreted later.


### Why this cell exists — diagnose missingness before cleaning for analysis

Missingness in this panel has several causes: true non-coverage, unresolved countries, no joined source row, sparse context indicators, or legitimate absence of observed violence.

This cell reports missingness in staging. It is not a command to fill everything; it is a diagnostic that helps preserve the distinction between `0` and `NA`.

In [23]:
missingness_variables = [
    col for col in staging_panel.columns
    if (
        col != "panel_key"
        and not col.endswith("_has_source_row")
        and not col.endswith("_has_any_source_row")
        and not col.endswith("_in_temporal_coverage")
        and not col.endswith("_country_in_source_universe")
        and not col.endswith("_can_zero_fill")
        and col != "analysis_core_country_universe"
    )
]

missingness_report = pl.DataFrame([
    {
        "variable": col,
        "missing_rows": int(staging_panel.select(pl.col(col).is_null().sum()).item()),
        "missing_pct": round(float(staging_panel.select(pl.col(col).is_null().mean()).item() * 100), 2),
    }
    for col in missingness_variables
]).sort(["missing_pct", "variable"], descending=[True, False])

missingness_report.head(25)


variable,missing_rows,missing_pct
str,i64,f64
"""ucdp_os_best_fatalities_allocated""",8528,89.41
"""ucdp_os_best_fatalities_full_multicountry""",8528,89.41
"""ucdp_os_government_actor_records""",8528,89.41
"""ucdp_os_high_fatalities_allocated""",8528,89.41
"""ucdp_os_low_fatalities_allocated""",8528,89.41
"""ucdp_os_multicountry_best_fatalities_allocated""",8528,89.41
"""ucdp_os_multicountry_records""",8528,89.41
"""ucdp_os_records""",8528,89.41
"""ucdp_ged_best_deaths""",7406,77.65


### Why this cell exists — create zero-filled metrics and derive analysis-ready columns

The staging panel contains raw source metrics. For selected conflict-count and intensity variables, the analysis-ready panel needs conditional zero-filled versions.

This cell applies the project rule: **fill with `0` only when the source is in temporal coverage, the country belongs to the source's harmonised country universe, and the joined metric is missing**. It then derives the final `analysis_ready` column list and writes a zero-fill impact report so this stricter rule remains auditable.


<a id="panel-outputs"></a>

<div class="cl-responsive cl-section-block" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 10</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">Staging and analysis-ready panels</h2>
  <p style='margin:8px 0 0 0;color:#D9E2EC;line-height:1.45;'>Separate auditability from analytical usability.</p>
</div>


The notebook now creates the two panel layers.

### Why two panels?

The staging panel and the analysis-ready panel serve different users and moments in the workflow.

| Panel | What it preserves | What it is for |
|---|---|---|
| `staging` | Detailed source variables, raw labels, diagnostics, flags and intermediate metrics. | Auditing, debugging, crosswalk review and methodological traceability. |
| `analysis_ready` | Curated variables, coverage flags, source-row flags and selected zero-filled metrics. | Notebook 03 descriptive country-year analysis and visualisation. |

The analysis-ready panel is therefore **not** a casual subset. It is a documented analytical view derived from staging.

### Zero-filling rule

Zero-filling is conditional, not automatic.

| Case | Analysis-ready value |
|---|---|
| Source covers the year and no source row is joined | `0` for selected conflict-count/intensity variables. |
| Source does not cover the year | `NA`. |
| Country could not be resolved | Not silently imputed; diagnose via crosswalk. |
| Contextual indicator unavailable | `NA`; never zero-filled. |

This is the main mechanism preserving the difference between observed absence and missing data.

### Column-selection table

The reduction from 69 staging columns to 43 analysis-ready columns is documented directly below as a simple markdown table.

| Treatment | Meaning |
|---|---|
| **Gardée** | The staging column is retained unchanged in `analysis_ready`. |
| **Remplacée par `_zf`** | The raw staging metric stays in staging, while a zero-filled version is exported for analysis. |
| **Staging only** | The column remains available for audit/debugging, but is intentionally excluded from the first analysis-ready view. |

`Staging only` does **not** mean discarded from the project. It means excluded from the first analytical view to keep the V1 panel interpretable.


### Staging → analysis-ready column policy

The final column decision is now generated by code and documented in the data dictionary. The policy is:

| Column family | Treatment | Reason |
|---|---|---|
| Panel keys (`country_iso3`, `year`, `panel_key`) | Kept unchanged | Required for all downstream joins and validations. |
| Analytical window and core-universe flags | Kept unchanged | Protects Notebook 03 from accidental use of out-of-window years or non-core country units in macro comparisons. |
| Source temporal coverage flags | Kept unchanged | Required to distinguish missing data from source non-coverage. |
| Source country-universe flags | Kept unchanged | Required to avoid false zeros for countries or territories outside a source's harmonised universe. |
| `*_can_zero_fill` flags | Kept unchanged | Makes the zero-fill eligibility rule explicit for every country-year. |
| Selected conflict metrics | Replaced by guarded `_zf` variables | Raw values remain in staging; analysis-ready receives zero-filled values only when temporal and country-universe conditions are both satisfied. |
| UCDP One-sided multi-country allocation diagnostics | Kept as guarded `_zf` variables where selected | Required for Axis B interpretation because some one-sided records are split across countries. |
| Source-row flags | Kept unchanged | Indicates whether a row was actually joined from the transformed source. |
| Context variables from V-Dem and WDI | Kept unchanged | Missing contextual values remain `NA`; they are never zero-filled. |
| Raw country labels, source regions, detailed dyad counts and uncertainty bounds | Staging only | Preserved for audit, robustness checks or Phase 2BIS, but excluded from the first descriptive panel to keep V1 interpretable. |

This avoids the maintenance risk of a long hand-written column table becoming stale after the final validation corrections. The machine-readable source of truth is `outputs/02_country_year_data_dictionary.csv`.


In [24]:
# zero fill
ZERO_FILL_RULES = {
    'ucdp_ov':['ucdp_ov_sb_dyad_count','ucdp_ov_sb_total_deaths_best','ucdp_ov_sb_deaths_civilians','ucdp_ov_ns_dyad_count','ucdp_ov_ns_total_deaths_best','ucdp_ov_ns_deaths_civilians','ucdp_ov_os_dyad_count','ucdp_ov_os_total_deaths_best','ucdp_ov_civilian_deaths_best','ucdp_ov_total_deaths_best','ucdp_ov_any_organized_violence'],
    'ucdp_ged':['ucdp_ged_events','ucdp_ged_best_deaths','ucdp_ged_high_deaths','ucdp_ged_low_deaths','ucdp_ged_civilian_deaths','ucdp_ged_state_based_events','ucdp_ged_non_state_events','ucdp_ged_one_sided_events','ucdp_ged_conflicts','ucdp_ged_dyads'],
    'ucdp_os':['ucdp_os_records','ucdp_os_best_fatalities_allocated','ucdp_os_low_fatalities_allocated','ucdp_os_high_fatalities_allocated','ucdp_os_best_fatalities_full_multicountry','ucdp_os_government_actor_records','ucdp_os_multicountry_records','ucdp_os_multicountry_best_fatalities_allocated'],
    'acled':['acled_political_violence_events','acled_reported_fatalities','acled_civilian_targeting_events','acled_reported_civilian_fatalities']
}

analysis_panel = staging_panel
zero_fill_impact_rows = []
excluded_universe_frames = []

for prefix, columns in ZERO_FILL_RULES.items():
    temporal_col = f'{prefix}_in_temporal_coverage'
    universe_col = f'{prefix}_analysis_unit_in_source_universe'
    range_col = f'{prefix}_analysis_unit_year_in_source_range'
    can_fill_col = f'{prefix}_can_zero_fill'

    excluded_universe_frames.append(
        analysis_panel
        .filter(pl.col('panel_year_in_recommended_window') & pl.col(temporal_col) & ~pl.col(universe_col))
        .group_by(['analysis_unit_id','analysis_unit_label','analysis_unit_type','country_iso3','country_name'])
        .agg([
            pl.len().alias('analysis_unit_years_not_zero_filled'),
            pl.col('year').min().alias('year_min'),
            pl.col('year').max().alias('year_max'),
        ])
        .with_columns([
            pl.lit(prefix).alias('source_prefix'),
            pl.lit(temporal_col).alias('temporal_coverage_flag'),
            pl.lit(universe_col).alias('analysis_unit_universe_flag'),
        ])
        .select(['source_prefix','analysis_unit_id','analysis_unit_label','analysis_unit_type','country_iso3','country_name','analysis_unit_years_not_zero_filled','year_min','year_max','temporal_coverage_flag','analysis_unit_universe_flag'])
    )

    for col in columns:
        if col in analysis_panel.columns:
            old_temporal_only_fill_expr = pl.col('panel_year_in_recommended_window') & pl.col(temporal_col) & pl.col(col).is_null()
            new_guarded_fill_expr = pl.col('panel_year_in_recommended_window') & pl.col(can_fill_col) & pl.col(col).is_null()
            old_temporal_only_filled = int(analysis_panel.select(old_temporal_only_fill_expr.sum()).item())
            new_guarded_filled = int(analysis_panel.select(new_guarded_fill_expr.sum()).item())
            zero_fill_impact_rows.append({
                'source_prefix': prefix,
                'variable': col,
                'old_temporal_only_zero_fills': old_temporal_only_filled,
                'new_analysis_unit_guarded_zero_fills': new_guarded_filled,
                'prevented_false_zero_fills': old_temporal_only_filled - new_guarded_filled,
            })
            analysis_panel = analysis_panel.with_columns(
                pl.when(pl.col(can_fill_col) & pl.col(col).is_null())
                .then(pl.lit(0))
                .otherwise(pl.col(col))
                .alias(f'{col}_zf')
            )

zero_fill_impact_report = pl.DataFrame(zero_fill_impact_rows).sort(['prevented_false_zero_fills','source_prefix','variable'], descending=[True,False,False])
zero_fill_excluded_analysis_unit_universe = pl.concat(excluded_universe_frames, how='vertical').sort(['source_prefix','analysis_unit_id'])

analysis_ready_columns = [
    'analysis_unit_id',
    'analysis_unit_label',
    'analysis_unit_type',
    'country_iso3',
    'country_name',
    'is_iso3_country',
    'is_non_iso_analysis_unit',
    'year',
    'panel_key',
    'panel_year_in_recommended_window',
    'analysis_core_macro_universe',
    'analysis_conflict_universe',
    'unit_exists_from_year',
    'unit_exists_in_year',
    'ucdp_ov_analysis_unit_in_source_universe',
    'ucdp_ged_analysis_unit_in_source_universe',
    'ucdp_os_analysis_unit_in_source_universe',
    'acled_analysis_unit_in_source_universe',
    'ucdp_ov_analysis_unit_year_in_source_range',
    'ucdp_ged_analysis_unit_year_in_source_range',
    'ucdp_os_analysis_unit_year_in_source_range',
    'acled_analysis_unit_year_in_source_range',
    'ucdp_ov_can_zero_fill',
    'ucdp_ged_can_zero_fill',
    'ucdp_os_can_zero_fill',
    'acled_can_zero_fill',
    'ucdp_ov_civilian_deaths_best_zf',
    'ucdp_ov_total_deaths_best_zf',
    'ucdp_ov_any_organized_violence_zf',
    'ucdp_ov_has_source_row',
    'ucdp_ged_events_zf',
    'ucdp_ged_best_deaths_zf',
    'ucdp_ged_civilian_deaths_zf',
    'ucdp_ged_state_based_events_zf',
    'ucdp_ged_non_state_events_zf',
    'ucdp_ged_one_sided_events_zf',
    'ucdp_ged_has_source_row',
    'ucdp_os_records_zf',
    'ucdp_os_best_fatalities_allocated_zf',
    'ucdp_os_multicountry_records_zf',
    'ucdp_os_multicountry_best_fatalities_allocated_zf',
    'ucdp_os_has_source_row',
    'acled_political_violence_events_zf',
    'acled_reported_fatalities_zf',
    'acled_civilian_targeting_events_zf',
    'acled_reported_civilian_fatalities_zf',
    'acled_has_any_source_row',
    'acled_year_is_partial',
    'vdem_electoral_democracy',
    'vdem_liberal_democracy',
    'vdem_rule_of_law',
    'vdem_political_corruption',
    'vdem_has_source_row',
    'wdi_gdp_per_capita_constant_2015_usd',
    'wdi_gini_index',
    'wdi_population_total',
    'wdi_secondary_school_enrollment_gross_pct',
    'wdi_under5_mortality_rate',
    'wdi_urban_population_pct_total',
    'wdi_has_source_row',
    'ucdp_ov_in_temporal_coverage',
    'ucdp_ged_in_temporal_coverage',
    'ucdp_os_in_temporal_coverage',
    'vdem_in_temporal_coverage',
    'wdi_in_temporal_coverage',
    'acled_in_temporal_coverage'
]

missing_analysis_columns = [col for col in analysis_ready_columns if col not in analysis_panel.columns]
if missing_analysis_columns:
    raise AssertionError(f"Analysis-ready columns missing from analysis panel: {missing_analysis_columns}")

analysis_ready_panel = (
    analysis_panel
    .filter(pl.col('panel_year_in_recommended_window'))
    .select(analysis_ready_columns)
)

selection_summary = pl.DataFrame([
    {"decision": "Kept unchanged", "n_columns": len([c for c in analysis_ready_columns if not c.endswith('_zf')])},
    {"decision": "Replaced by a guarded _zf variable", "n_columns": len([c for c in analysis_ready_columns if c.endswith('_zf')])},
    {"decision": "Staging only", "n_columns": len([c for c in staging_panel.columns if c not in analysis_ready_columns])},
])

selection_summary

decision,n_columns
str,i64
"""Kept unchanged""",49
"""Replaced by a guarded _zf variable""",17
"""Staging only""",51


### What this output tells us

The selection summary is the compact bridge from staging to analysis-ready: some columns are kept, selected conflict metrics are replaced by `_zf` versions, and detailed technical columns remain staging-only.

The complete 69-row decision table is written directly in markdown below, so the reader can inspect every column-level choice without reading implementation code.


### Staging → analysis-ready column decisions

The detailed column-level source of truth is generated in `outputs/02_country_year_data_dictionary.csv`. The notebook keeps the human-readable policy above instead of maintaining a long hand-written table that can drift after methodological corrections.


### What the column policy tells us

The reduction from staging to analysis-ready remains auditable:

- **Kept unchanged** columns are necessary for keys, interpretation, source coverage, country-universe eligibility or context;
- **guarded `_zf`** columns keep the raw metric in staging but expose a conditional zero-filled version for analysis;
- **Staging only** columns are retained for audit, diagnostics, uncertainty review or possible Phase 2BIS enrichment, but are not part of the first descriptive panel.

The important methodological point is that excluded columns are not lost. They remain available in `02_country_year_panel_staging` and can be reintroduced later if Notebook 03 or Phase 2BIS requires them.


### Why this cell exists — compare the two panel layers

After documenting the column-level decisions, we need one compact check showing what those decisions produce in practice: how large the staging panel is, how large the analysis-ready panel is, and which years each one covers.

This makes the consequence of the policy visible before exporting files.

In [25]:
panel_layer_summary = pl.DataFrame([
    {
        "panel": "staging",
        "rows": staging_panel.height,
        "columns": staging_panel.width,
        "analysis_units": int(staging_panel.select(pl.col("analysis_unit_id").n_unique()).item()),
        "non_iso_analysis_units": int(staging_panel.filter(pl.col("is_non_iso_analysis_unit")).select(pl.col("analysis_unit_id").n_unique()).item()),
        "year_min": int(staging_panel.select(pl.col("year").min()).item()),
        "year_max": int(staging_panel.select(pl.col("year").max()).item()),
        "role": "Audit layer: complete build window, detailed source diagnostics and staging-only variables.",
    },
    {
        "panel": "analysis_ready",
        "rows": analysis_ready_panel.height,
        "columns": analysis_ready_panel.width,
        "analysis_units": int(analysis_ready_panel.select(pl.col("analysis_unit_id").n_unique()).item()),
        "non_iso_analysis_units": int(analysis_ready_panel.filter(pl.col("is_non_iso_analysis_unit")).select(pl.col("analysis_unit_id").n_unique()).item()),
        "year_min": int(analysis_ready_panel.select(pl.col("year").min()).item()),
        "year_max": int(analysis_ready_panel.select(pl.col("year").max()).item()),
        "role": "Interpretation layer: recommended window, guarded zero-filled metrics and curated variables for Notebook 03.",
    },
])

panel_layer_summary

panel,rows,columns,analysis_units,non_iso_analysis_units,year_min,year_max,role
str,i64,i64,i64,i64,i64,i64,str
"""staging""",9538,100,251,5,1989,2026,"""Audit layer: complete build window, detailed source diagnostics and staging-only variables."""
"""analysis_ready""",9287,66,251,5,1989,2025,"""Interpretation layer: recommended window, guarded zero-filled metrics and curated variables for Notebook 03."""


### What this comparison tells us

The analysis-ready panel should be smaller in both rows and columns. Fewer rows come from filtering to the recommended analysis window; fewer columns come from the documented selection policy.

This confirms the intended separation of responsibilities: staging remains the audit layer, while analysis-ready becomes the interpretation layer for the next notebook.

<a id="data-dictionary"></a>

<div class="cl-responsive cl-section-block" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 11</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">Data dictionary and exports</h2>
  <p style='margin:8px 0 0 0;color:#D9E2EC;line-height:1.45;'>Variable documentation and persisted outputs.</p>
</div>


The data dictionary and the column-policy narrative answer different questions.

| Documentation object | Question answered |
|---|---|
| Data dictionary | What does each exported variable mean, and how should missing values / zeros be interpreted? |
| Column-policy narrative | Why are variables kept unchanged, guarded with `_zf`, or left in staging? |
| Join report | How did each raw source contribute to the final panel? |
| Crosswalk unmatched | Are any country labels still unresolved after V1 arbitration? It should be empty in this build. |
| Zero-fill impact report | How many potential temporal-only zeros were prevented by the stricter source-country-universe guard? |
| Crosswalk arbitration impact report | What is the empirical footprint of the V1 mapping/exclusion/outside-ISO3 decisions? |

Together, these documents make the panel reusable without forcing the next reader to reverse-engineer the notebook.


### Why this cell exists — document variables for reuse

The panel should be reusable without requiring the next reader to reverse-engineer the code. The data dictionary documents what each exported variable means and how missing values and zeros should be interpreted.

This cell builds the dictionary for staging-only variables, analysis-ready variables and derived guarded `_zf` metrics.


In [26]:
# dictionary
VARIABLE_DESCRIPTIONS = {
    'analysis_unit_id': ('Panel', 'Primary analytical unit key. ISO3 countries use their ISO3 code; validated non-ISO historical/composite units use a dedicated stable ID such as HIST_SERBIA_YUGOSLAVIA.', 'Never missing after backbone construction.', 'Not applicable.'),
    'analysis_unit_label': ('Panel', 'Readable label for the analytical unit.', 'Never intentionally missing after backbone construction.', 'Not applicable.'),
    'analysis_unit_type': ('Panel', 'Type of analysis unit: iso3_country, historical_entity, historical_composite_entity, or another explicitly documented unit type.', 'Never intentionally missing after backbone construction.', 'Not applicable.'),
    'country_iso3': ('Panel', 'ISO 3166-1 alpha-3 country code when the analysis unit is a valid ISO3 country or accepted special code. Null for non-ISO historical/composite analysis units.', 'Missing is expected for non-ISO analysis units and should not be interpreted as missing data.', 'Not applicable.'),
    'country_name': ('Panel', 'Readable ISO country name attached to country_iso3 when applicable.', 'Missing is expected for non-ISO analysis units.', 'Not applicable.'),
    'is_iso3_country': ('Panel', 'Whether the analysis unit is an ISO3-compatible country/statistical unit.', 'Boolean flag, never intentionally missing.', 'Not applicable.'),
    'is_non_iso_analysis_unit': ('Panel', 'Whether the analysis unit is retained outside ISO3 to avoid arbitrary remapping or data loss.', 'Boolean flag, never intentionally missing.', 'Not applicable.'),
    'year': ('Panel', 'Calendar year.', 'Never missing after backbone construction.', 'Not applicable.'),
    'panel_key': ('Panel', 'Stable analysis-unit-year key built as analysis_unit_id + _ + year.', 'Never missing after backbone construction.', 'Not applicable.'),
    'panel_year_in_recommended_window': ('Panel', 'Flag for the diagnostic-based recommended analysis window.', 'Boolean flag, never intentionally missing.', 'Not applicable.'),
    'analysis_core_macro_universe': ('Panel', 'Recommended macro-context universe for Notebook 03 Axis A. True only for ISO3-compatible units with standard country-level context coverage.', 'False means the row remains available but should not be used by default in core macro-context comparisons.', 'Not applicable.'),
    'analysis_conflict_universe': ('Panel', 'Recommended conflict/civilian-exposure universe for Notebook 03 Axis B. Includes ISO3 and retained non-ISO units with real conflict-source signal.', 'False means the row has no conflict-source unit membership in the current build.', 'Not applicable.'),
    'unit_exists_from_year': ('Panel', 'First year in which the analysis unit exists as a sovereign/independent unit; defaults to the staging-window start.', 'Never missing after backbone construction.', 'Not applicable.'),
    'unit_exists_in_year': ('Panel', 'True when the analysis unit exists in the given year. This bounds conflict zero-filling and prevents synthetic zeros before unit existence.', 'Boolean flag, never missing.', 'Controls conflict _zf variables.'),
}
for prefix, source_name in [('ucdp_ov','UCDP Organized Violence CY'),('ucdp_ged','UCDP GED aggregated to analysis-unit-year'),('ucdp_os','UCDP One-sided transformed to analysis-unit-year'),('acled','ACLED analysis-unit-year summaries'),('vdem','V-Dem selected context'),('wdi','WDI selected context')]:
    VARIABLE_DESCRIPTIONS[f'{prefix}_in_temporal_coverage'] = (source_name, 'Source-level temporal coverage flag.', 'Boolean flag, never intentionally missing.', 'Not applicable.')
for prefix, source_name in [('ucdp_ov','UCDP Organized Violence CY'),('ucdp_ged','UCDP GED aggregated to analysis-unit-year'),('ucdp_os','UCDP One-sided transformed to analysis-unit-year'),('acled','ACLED analysis-unit-year summaries')]:
    VARIABLE_DESCRIPTIONS[f'{prefix}_analysis_unit_in_source_universe'] = (source_name, 'Whether the analysis unit appears in the harmonised unit universe for this source after conservative matching and V1 arbitration.', 'False means the source should not be interpreted as observing a zero for this unit by default.', 'Not applicable.')
    VARIABLE_DESCRIPTIONS[f'{prefix}_analysis_unit_year_in_source_range'] = (source_name, 'For ISO3 units this is true by design; for retained non-ISO units this restricts zero-filling to the observed source-label year range.', 'False prevents non-ISO historical/composite units from receiving synthetic zeros outside their observed source range.', 'Controls guarded _zf variables.')
    VARIABLE_DESCRIPTIONS[f'{prefix}_can_zero_fill'] = (source_name, 'Eligibility flag for zero-filling selected conflict metrics: source temporal coverage AND analysis-unit source-universe membership AND valid unit-year range AND unit existence in year.', 'False means missing selected conflict metrics remain NA rather than becoming 0.', 'Controls guarded _zf variables.')
for prefix in ['ucdp_ov','ucdp_ged','ucdp_os','acled']:
    VARIABLE_DESCRIPTIONS[f'{prefix}_analysis_unit_year_min'] = ('Source diagnostics', 'First observed year for this analysis unit in the harmonised source panel.', 'May be missing when the unit is outside that source universe.', 'Not applicable.')
    VARIABLE_DESCRIPTIONS[f'{prefix}_analysis_unit_year_max'] = ('Source diagnostics', 'Last observed year for this analysis unit in the harmonised source panel.', 'May be missing when the unit is outside that source universe.', 'Not applicable.')

for col in staging_panel.columns:
    if col in VARIABLE_DESCRIPTIONS:
        continue
    if col.endswith('_has_source_row') or col.endswith('_has_any_source_row'):
        VARIABLE_DESCRIPTIONS[col] = ('Source diagnostics', 'Whether a source row was joined for this analysis-unit-year. The exact interpretation varies by source and should be read with temporal and analysis-unit-universe flags.', 'False means no joined row after harmonisation; it is not by itself sufficient to infer zero.', 'Not applicable.')
    elif col.startswith('ucdp_ov_'):
        VARIABLE_DESCRIPTIONS[col] = ('UCDP Organized Violence CY', 'Raw or diagnostic UCDP Organized Violence analysis-unit-year variable.', 'Missing in staging means no joined raw source row, unresolved mapping, or inapplicable unit.', 'Zero is not imputed in staging.')
    elif col.startswith('ucdp_ged_'):
        VARIABLE_DESCRIPTIONS[col] = ('UCDP GED', 'Analysis-unit-year aggregate derived from UCDP GED event records.', 'Missing in staging means no joined event aggregate, unresolved mapping, or inapplicable unit.', 'Zero is not imputed in staging.')
    elif col.startswith('ucdp_os_'):
        VARIABLE_DESCRIPTIONS[col] = ('UCDP One-sided', 'Analysis-unit-year aggregate derived from UCDP One-sided records; multi-unit locations use an allocated metric where indicated.', 'Missing in staging means no joined one-sided aggregate, unresolved mapping, or inapplicable unit.', 'Zero is not imputed in staging.')
    elif col.startswith('acled_'):
        VARIABLE_DESCRIPTIONS[col] = ('ACLED', 'ACLED benchmark variable or diagnostic flag.', 'Missing in staging means no joined ACLED row or unresolved/outside-scope unit mapping.', 'Zero is not imputed in staging.')
    elif col.startswith('vdem_'):
        VARIABLE_DESCRIPTIONS[col] = ('V-Dem', 'Selected institutional or political context variable.', 'Missing remains analytically meaningful and is not zero-filled.', 'Not applicable.')
    elif col.startswith('wdi_'):
        VARIABLE_DESCRIPTIONS[col] = ('WDI', 'Selected socioeconomic or demographic context variable.', 'Missing remains analytically meaningful and is not zero-filled.', 'Not applicable.')
    else:
        VARIABLE_DESCRIPTIONS[col] = ('Panel', 'Panel construction variable.', 'See source flag and join report.', 'Not applicable.')
for prefix, columns in ZERO_FILL_RULES.items():
    for col in columns:
        z = f'{col}_zf'
        if z in analysis_ready_panel.columns:
            fam, desc, _, _ = VARIABLE_DESCRIPTIONS.get(col, ('Conflict metrics','Zero-filled analysis metric.','',''))
            VARIABLE_DESCRIPTIONS[z] = (fam, desc + ' Guarded zero-filled analysis variable.', 'NA remains outside source temporal coverage, outside source analysis-unit universe, outside valid unit-year range, before unit existence, or where the source is not applicable.', '0 means no joined event/source row inside source temporal coverage, source analysis-unit universe, valid unit-year range, and unit existence.')
all_vars=[]
for c in staging_panel.columns + analysis_ready_panel.columns:
    if c not in all_vars:
        all_vars.append(c)
data_dictionary = pl.DataFrame([
    {
        'variable': v,
        'stage': ('both' if v in staging_panel.columns and v in analysis_ready_panel.columns else 'staging' if v in staging_panel.columns else 'analysis_ready'),
        'dtype_staging': str(staging_panel.schema.get(v)) if v in staging_panel.columns else None,
        'dtype_analysis_ready': str(analysis_ready_panel.schema.get(v)) if v in analysis_ready_panel.columns else None,
        'source_family': VARIABLE_DESCRIPTIONS.get(v, ('Unknown','','',''))[0],
        'description': VARIABLE_DESCRIPTIONS.get(v, ('','Variable added by panel construction.','',''))[1],
        'missing_value_rule': VARIABLE_DESCRIPTIONS.get(v, ('','','See join report and source flags.',''))[2],
        'zero_rule': VARIABLE_DESCRIPTIONS.get(v, ('','','','Not applicable.'))[3],
    }
    for v in all_vars
])

data_dictionary.head(25)

variable,stage,dtype_staging,dtype_analysis_ready,source_family,description,missing_value_rule,zero_rule
str,str,str,str,str,str,str,str
"""analysis_unit_id""","""both""","""String""","""String""","""Panel""","""Primary analytical unit key. ISO3 countries use their ISO3 code; validated non-ISO historical/composite units use a dedi…","""Never missing after backbone construction.""","""Not applicable."""
"""analysis_unit_label""","""both""","""String""","""String""","""Panel""","""Readable label for the analytical unit.""","""Never intentionally missing after backbone construction.""","""Not applicable."""
"""analysis_unit_type""","""both""","""String""","""String""","""Panel""","""Type of analysis unit: iso3_country, historical_entity, historical_composite_entity, or another explicitly documented un…","""Never intentionally missing after backbone construction.""","""Not applicable."""
"""country_iso3""","""both""","""String""","""String""","""Panel""","""ISO 3166-1 alpha-3 country code when the analysis unit is a valid ISO3 country or accepted special code. Null for non-IS…","""Missing is expected for non-ISO analysis units and should not be interpreted as missing data.""","""Not applicable."""
"""country_name""","""both""","""String""","""String""","""Panel""","""Readable ISO country name attached to country_iso3 when applicable.""","""Missing is expected for non-ISO analysis units.""","""Not applicable."""
"""is_iso3_country""","""both""","""Boolean""","""Boolean""","""Panel""","""Whether the analysis unit is an ISO3-compatible country/statistical unit.""","""Boolean flag, never intentionally missing.""","""Not applicable."""
"""is_non_iso_analysis_unit""","""both""","""Boolean""","""Boolean""","""Panel""","""Whether the analysis unit is retained outside ISO3 to avoid arbitrary remapping or data loss.""","""Boolean flag, never intentionally missing.""","""Not applicable."""
"""year""","""both""","""Int64""","""Int64""","""Panel""","""Calendar year.""","""Never missing after backbone construction.""","""Not applicable."""
"""panel_key""","""both""","""String""","""String""","""Panel""","""Stable analysis-unit-year key built as analysis_unit_id + _ + year.""","""Never missing after backbone construction.""","""Not applicable."""


### What this output tells us

The data dictionary is the machine-readable documentation layer for Notebook 03. It should explicitly describe source temporal coverage, country-universe flags, `*_can_zero_fill` flags and guarded `_zf` metrics so that downstream analysis does not treat every missing value or zero as equivalent.


### Why this cell exists — write durable artefacts

The notebook is not complete until the panel and its audit files exist on disk. Notebook 03 should consume persisted outputs, not hidden in-memory state from this notebook.

This cell writes the staging panel, analysis-ready panel, data dictionary, join report and crosswalk diagnostic to `outputs/`.

In [27]:
staging_parquet_path = OUTPUT_DIR / "02_analysis_unit_year_panel_staging.parquet"
staging_csv_path = OUTPUT_DIR / "02_analysis_unit_year_panel_staging.csv"
analysis_parquet_path = OUTPUT_DIR / "02_analysis_unit_year_panel_analysis_ready.parquet"
analysis_csv_path = OUTPUT_DIR / "02_analysis_unit_year_panel_analysis_ready.csv"
data_dictionary_path = OUTPUT_DIR / "02_analysis_unit_year_data_dictionary.csv"
join_report_path = OUTPUT_DIR / "02_analysis_unit_year_join_report.csv"
zero_fill_impact_path = OUTPUT_DIR / "02_zero_fill_impact_report.csv"
zero_fill_excluded_universe_path = OUTPUT_DIR / "02_zero_fill_excluded_analysis_unit_universe.csv"
crosswalk_arbitration_impact_path = OUTPUT_DIR / "02_crosswalk_arbitration_impact_report.csv"
ucdp_os_multicountry_allocation_path = OUTPUT_DIR / "02_ucdp_os_multicountry_allocation_report.csv"

# Legacy aliases are written too, so earlier notebook references fail less abruptly.
legacy_staging_parquet_path = OUTPUT_DIR / "02_country_year_panel_staging.parquet"
legacy_staging_csv_path = OUTPUT_DIR / "02_country_year_panel_staging.csv"
legacy_analysis_parquet_path = OUTPUT_DIR / "02_country_year_panel_analysis_ready.parquet"
legacy_analysis_csv_path = OUTPUT_DIR / "02_country_year_panel_analysis_ready.csv"
legacy_data_dictionary_path = OUTPUT_DIR / "02_country_year_data_dictionary.csv"
legacy_join_report_path = OUTPUT_DIR / "02_country_year_join_report.csv"

for path in [staging_parquet_path, legacy_staging_parquet_path]:
    staging_panel.write_parquet(path)
for path in [staging_csv_path, legacy_staging_csv_path]:
    staging_panel.write_csv(path)
for path in [analysis_parquet_path, legacy_analysis_parquet_path]:
    analysis_ready_panel.write_parquet(path)
for path in [analysis_csv_path, legacy_analysis_csv_path]:
    analysis_ready_panel.write_csv(path)
for path in [data_dictionary_path, legacy_data_dictionary_path]:
    data_dictionary.write_csv(path)
for path in [join_report_path, legacy_join_report_path]:
    join_report.write_csv(path)

zero_fill_impact_report.write_csv(zero_fill_impact_path)
zero_fill_excluded_analysis_unit_universe.write_csv(zero_fill_excluded_universe_path)
crosswalk_arbitration_impact_report.write_csv(crosswalk_arbitration_impact_path)
ucdp_os_multicountry_allocation_report.write_csv(ucdp_os_multicountry_allocation_path)
# Rewrite crosswalk audit artefacts here for reproducibility after any rerun.
country_crosswalk_arbitration_v1.write_csv(crosswalk_arbitration_path)
country_crosswalk_unmatched.write_csv(crosswalk_unmatched_path)

export_manifest = pl.DataFrame([
    {"output": staging_parquet_path.relative_to(PROJECT_ROOT).as_posix(), "rows": staging_panel.height, "columns": staging_panel.width, "role": "Auditable full analysis-unit-year panel"},
    {"output": staging_csv_path.relative_to(PROJECT_ROOT).as_posix(), "rows": staging_panel.height, "columns": staging_panel.width, "role": "CSV mirror of the staging panel"},
    {"output": analysis_parquet_path.relative_to(PROJECT_ROOT).as_posix(), "rows": analysis_ready_panel.height, "columns": analysis_ready_panel.width, "role": "Curated analysis-unit-year panel for Notebook 03"},
    {"output": analysis_csv_path.relative_to(PROJECT_ROOT).as_posix(), "rows": analysis_ready_panel.height, "columns": analysis_ready_panel.width, "role": "CSV mirror of the analysis-ready panel"},
    {"output": data_dictionary_path.relative_to(PROJECT_ROOT).as_posix(), "rows": data_dictionary.height, "columns": data_dictionary.width, "role": "Variable-level documentation"},
    {"output": join_report_path.relative_to(PROJECT_ROOT).as_posix(), "rows": join_report.height, "columns": join_report.width, "role": "Source transformation and join audit"},
    {"output": zero_fill_impact_path.relative_to(PROJECT_ROOT).as_posix(), "rows": zero_fill_impact_report.height, "columns": zero_fill_impact_report.width, "role": "Zero-fill guard impact audit"},
    {"output": zero_fill_excluded_universe_path.relative_to(PROJECT_ROOT).as_posix(), "rows": zero_fill_excluded_analysis_unit_universe.height, "columns": zero_fill_excluded_analysis_unit_universe.width, "role": "Analysis units deliberately excluded from source-universe zero-filling"},
    {"output": crosswalk_arbitration_impact_path.relative_to(PROJECT_ROOT).as_posix(), "rows": crosswalk_arbitration_impact_report.height, "columns": crosswalk_arbitration_impact_report.width, "role": "Quantified impact of V1 country arbitration decisions"},
    {"output": ucdp_os_multicountry_allocation_path.relative_to(PROJECT_ROOT).as_posix(), "rows": ucdp_os_multicountry_allocation_report.height, "columns": ucdp_os_multicountry_allocation_report.width, "role": "UCDP One-sided multi-country allocation diagnostic"},
    {"output": crosswalk_arbitration_path.relative_to(PROJECT_ROOT).as_posix(), "rows": country_crosswalk_arbitration_v1.height, "columns": country_crosswalk_arbitration_v1.width, "role": "Validated V1 arbitration table"},
    {"output": crosswalk_unmatched_path.relative_to(PROJECT_ROOT).as_posix(), "rows": country_crosswalk_unmatched.height, "columns": country_crosswalk_unmatched.width, "role": "Remaining unresolved labels after V1 arbitration"},
    {"output": legacy_analysis_csv_path.relative_to(PROJECT_ROOT).as_posix(), "rows": analysis_ready_panel.height, "columns": analysis_ready_panel.width, "role": "Legacy filename alias containing the same analysis-unit-year data model"},
])

export_manifest

output,rows,columns,role
str,i64,i64,str
"""outputs/02_analysis_unit_year_panel_staging.parquet""",9538,100,"""Auditable full analysis-unit-year panel"""
"""outputs/02_analysis_unit_year_panel_staging.csv""",9538,100,"""CSV mirror of the staging panel"""
"""outputs/02_analysis_unit_year_panel_analysis_ready.parquet""",9287,66,"""Curated analysis-unit-year panel for Notebook 03"""
"""outputs/02_analysis_unit_year_panel_analysis_ready.csv""",9287,66,"""CSV mirror of the analysis-ready panel"""
"""outputs/02_analysis_unit_year_data_dictionary.csv""",117,8,"""Variable-level documentation"""
"""outputs/02_analysis_unit_year_join_report.csv""",9,16,"""Source transformation and join audit"""
"""outputs/02_zero_fill_impact_report.csv""",33,5,"""Zero-fill guard impact audit"""
"""outputs/02_zero_fill_excluded_analysis_unit_universe.csv""",334,11,"""Analysis units deliberately excluded from source-universe zero-filling"""
"""outputs/02_crosswalk_arbitration_impact_report.csv""",46,23,"""Quantified impact of V1 country arbitration decisions"""


### What this output tells us

The export manifest confirms that all expected artefacts were written. It also makes the practical handoff clear: Notebook 03 should normally read the analysis-ready Parquet file, while audit work should refer back to staging, join report, data dictionary and crosswalk unmatched.

<a id="validation-handoff"></a>

<div class="cl-responsive cl-section-block" style="box-sizing:border-box;width:100%;max-width:100%;overflow-x:hidden;white-space:normal;overflow-wrap:anywhere;word-break:normal;margin:44px 0 24px 0;padding:18px 22px;border-radius:10px;background:#102A43;color:white;">
  <div style="font-size:13px;letter-spacing:.12em;text-transform:uppercase;color:#D4A44C;font-weight:700;">Section 12</div>
  <h2 style="margin:6px 0 0 0;color:white;overflow-wrap:anywhere;word-break:normal;">Validation and handoff</h2>
  <p style='margin:8px 0 0 0;color:#D9E2EC;line-height:1.45;'>Final quality checks and what remains open.</p>
</div>


The final validation section checks the structural guarantees that Notebook 03 will rely on.

The most important checks are:

- one unique row per `analysis_unit_id + year` in both panels;
- no missing panel keys;
- analysis-ready columns exist after zero-filling and selection;
- all required outputs are written;
- unresolved country mappings are absent after V1 arbitration;
- selected conflict metrics are not zero-filled outside the relevant source country universe;
- 2026 is absent from `analysis_ready`.

These checks do not prove that every substantive interpretation is correct. They prove that the panel is internally consistent, auditable and protected against the main methodological risks identified during final review.


### Why this cell exists — validate the panel contract

Before handing off to analysis, the notebook must prove that its structural promises hold: unique country-year keys, complete documentation coverage, required files written, and unresolved country labels exported.

This cell builds a validation checklist with pass/fail status for the guarantees Notebook 03 will depend on.

In [28]:
validation_checks = []


def add_check(name: str, passed: bool, detail: str) -> None:
    validation_checks.append({"check": name, "passed": bool(passed), "detail": detail})


add_check(
    "staging_unique_analysis_unit_year_key",
    duplicate_key_count(staging_panel) == 0,
    "Staging panel has one row per analysis_unit_id + year.",
)
add_check(
    "analysis_ready_unique_analysis_unit_year_key",
    duplicate_key_count(analysis_ready_panel) == 0,
    "Analysis-ready panel has one row per analysis_unit_id + year.",
)
add_check(
    "staging_no_missing_analysis_unit_keys",
    int(staging_panel.select(pl.any_horizontal([pl.col("analysis_unit_id").is_null(), pl.col("year").is_null()]).sum()).item()) == 0,
    "Staging panel has no missing analysis-unit keys.",
)
add_check(
    "analysis_ready_no_missing_analysis_unit_keys",
    int(analysis_ready_panel.select(pl.any_horizontal([pl.col("analysis_unit_id").is_null(), pl.col("year").is_null()]).sum()).item()) == 0,
    "Analysis-ready panel has no missing analysis-unit keys.",
)
add_check(
    "country_iso3_nullable_for_non_iso_units",
    int(analysis_ready_panel.filter(pl.col('is_non_iso_analysis_unit')).select(pl.col('country_iso3').is_not_null().sum()).item()) == 0,
    "Non-ISO analysis units remain outside country_iso3 rather than being force-mapped.",
)
add_check(
    "analysis_ready_columns_exist_in_analysis_panel",
    set(analysis_ready_panel.columns).issubset(set(analysis_panel.columns)),
    "Every analysis-ready column exists after zero-filling and column selection.",
)
add_check(
    "expected_exports_exist",
    all(path.exists() for path in [
        staging_parquet_path, staging_csv_path,
        analysis_parquet_path, analysis_csv_path,
        data_dictionary_path, join_report_path,
        crosswalk_arbitration_path, crosswalk_unmatched_path,
        zero_fill_impact_path, zero_fill_excluded_universe_path,
        crosswalk_arbitration_impact_path, ucdp_os_multicountry_allocation_path,
    ]),
    "All required outputs exist.",
)
add_check(
    "crosswalk_arbitration_exported",
    crosswalk_arbitration_path.exists() and country_crosswalk_arbitration_v1.height > 0,
    "The validated V1 arbitration table is exported.",
)
add_check(
    "no_remaining_unresolved_crosswalk_rows",
    country_crosswalk_unmatched.height == 0,
    "All initially unmatched crosswalk rows are mapped, excluded, or retained as non-ISO analysis units by the V1 arbitration.",
)

false_zero_checks = []
for prefix, columns in ZERO_FILL_RULES.items():
    universe_col = f'{prefix}_analysis_unit_in_source_universe'
    temporal_col = f'{prefix}_in_temporal_coverage'
    for col in columns:
        z_col = f'{col}_zf'
        if col in analysis_panel.columns and z_col in analysis_panel.columns:
            false_zero_count = int(analysis_panel.select((
                pl.col('panel_year_in_recommended_window')
                & pl.col(temporal_col)
                & ~pl.col(universe_col)
                & pl.col(col).is_null()
                & (pl.col(z_col) == 0)
            ).sum()).item())
            false_zero_checks.append(false_zero_count)

add_check(
    "no_guarded_zero_fill_outside_source_analysis_unit_universe",
    sum(false_zero_checks) == 0,
    "No selected conflict metric is zero-filled outside its source analysis-unit universe.",
)
add_check(
    "zero_fill_impact_report_non_empty",
    zero_fill_impact_report.height > 0,
    "Zero-fill impact report was generated.",
)
add_check(
    "crosswalk_arbitration_impact_report_non_empty",
    crosswalk_arbitration_impact_report.height > 0,
    "Crosswalk arbitration impact report was generated.",
)
add_check(
    "analysis_ready_excludes_2026",
    int(analysis_ready_panel.select((pl.col('year') == 2026).sum()).item()) == 0,
    "Analysis-ready excludes 2026 because core sources are not all complete for that year.",
)
add_check(
    "non_iso_analysis_units_preserved",
    int(analysis_ready_panel.filter(pl.col('is_non_iso_analysis_unit')).select(pl.col('analysis_unit_id').n_unique()).item()) > 0,
    "Non-ISO historical/composite units are retained in the analysis-ready panel.",
)
add_check(
    "serbia_yugoslavia_preserved_in_conflict_universe",
    int(analysis_ready_panel.filter(pl.col('analysis_unit_id') == 'HIST_SERBIA_YUGOSLAVIA').height) > 0
    and float(analysis_ready_panel.filter(pl.col('analysis_unit_id') == 'HIST_SERBIA_YUGOSLAVIA').select(pl.col('ucdp_ged_events_zf').sum()).item() or 0) > 0,
    "Serbia (Yugoslavia) is retained as a non-ISO analysis unit with GED conflict signal.",
)

existence_zero_violations = 0
for prefix, columns in ZERO_FILL_RULES.items():
    for col in columns:
        z = f'{col}_zf'
        if z in analysis_panel.columns:
            existence_zero_violations += int(analysis_panel.select(
                ((~pl.col('unit_exists_in_year')) & (pl.col(z) == 0) & pl.col(col).is_null()).sum()
            ).item())
add_check(
    'no_zero_fill_before_unit_exists',
    existence_zero_violations == 0,
    "No conflict metric is zero-filled before the analysis unit's existence year.",
)

validation_report = pl.DataFrame(validation_checks)
validation_report

check,passed,detail
str,bool,str
"""staging_unique_analysis_unit_year_key""",true,"""Staging panel has one row per analysis_unit_id + year."""
"""analysis_ready_unique_analysis_unit_year_key""",true,"""Analysis-ready panel has one row per analysis_unit_id + year."""
"""staging_no_missing_analysis_unit_keys""",true,"""Staging panel has no missing analysis-unit keys."""
"""analysis_ready_no_missing_analysis_unit_keys""",true,"""Analysis-ready panel has no missing analysis-unit keys."""
"""country_iso3_nullable_for_non_iso_units""",true,"""Non-ISO analysis units remain outside country_iso3 rather than being force-mapped."""
"""analysis_ready_columns_exist_in_analysis_panel""",true,"""Every analysis-ready column exists after zero-filling and column selection."""
"""expected_exports_exist""",true,"""All required outputs exist."""
"""crosswalk_arbitration_exported""",true,"""The validated V1 arbitration table is exported."""
"""no_remaining_unresolved_crosswalk_rows""",true,"""All initially unmatched crosswalk rows are mapped, excluded, or retained as non-ISO analysis units by the V1 arbitration…"


### What this output tells us

The validation report should contain only passing checks. These checks are structural and methodological guardrails: they prove that the panel is internally consistent, that V1 crosswalk arbitration is no longer unresolved, and that the zero-fill rule does not create false zeros outside the source country universe.


### Why this cell exists — fail loudly if validation fails

The previous cell assembled the validation checks. This cell turns those checks into an execution gate: if any structural guarantee fails, the notebook stops instead of exporting a silently broken panel.

In [29]:
if not validation_report.select(pl.col("passed").all()).item():
    failed = validation_report.filter(~pl.col("passed"))
    raise AssertionError("Validation failed:\n" + str(failed))

display_note(
    "<strong>Execution insight.</strong> Notebook 02 completed successfully with a Polars-first implementation and the validated V1 country crosswalk arbitration. "
    f"The staging panel contains <code>{staging_panel.height:,}</code> rows and <code>{staging_panel.width}</code> columns. "
    f"The analysis-ready panel contains <code>{analysis_ready_panel.height:,}</code> rows and <code>{analysis_ready_panel.width}</code> columns for "
    f"<code>{RECOMMENDED_YEAR_START}–{RECOMMENDED_YEAR_END}</code>. "
    f"Remaining unresolved crosswalk rows after arbitration: <code>{country_crosswalk_unmatched.height}</code>. "
    "The next notebook can now focus on descriptive analysis-unit-year analysis, using the exported source-universe, macro-universe and conflict-universe flags to keep the interpretation disciplined."
)

### Consequence for handoff

If this cell runs without raising an error, the notebook has satisfied its construction contract. The final handoff can therefore focus on what Notebook 03 should analyze and which decisions remain open, rather than on debugging panel structure.

## Handoff to Notebook 03

Notebook 02 establishes the country-year analytical foundation and now includes the validated V1 country crosswalk arbitration plus the final zero-fill and historical-entity safeguards.

Recommended sequence for Notebook 03:

1. Use `outputs/02_country_year_panel_analysis_ready.parquet` as the default input for descriptive country-year analysis.
2. Keep `outputs/02_country_year_panel_staging.parquet` available for audit and debugging.
3. Use `analysis_core_country_universe` as the default filter for macro-context comparisons unless the analysis explicitly needs the full extended backbone.
4. Use `*_can_zero_fill` and `*_country_in_source_universe` flags when interpreting zero-filled conflict metrics.
5. Keep `outputs/02_country_crosswalk_arbitration_v1.csv` as the methodological record for country harmonisation decisions.
6. Treat `outputs/02_country_crosswalk_unmatched.csv` as a residual quality gate. In this V1 build it should be empty after arbitration.
7. Review `outputs/02_crosswalk_arbitration_impact_report.csv` before interpreting historically complex cases, especially `Serbia (Yugoslavia)`.
8. Review `outputs/02_zero_fill_impact_report.csv` if a Notebook 03 result depends strongly on zero-filled conflict counts.
9. Use `outputs/02_ucdp_os_multicountry_allocation_report.csv` when working on Axis B and civilian exposure metrics.
10. Structure the first analyses around the two agreed axes:
   - Axis A — macro-context conflict vulnerability;
   - Axis B — civilian exposure and violence against civilians.
11. Keep EPR, SVAC, advanced ACLED metrics and broader V-Dem/WDI enrichment for Phase 2BIS after the minimal panel has been validated.

The main methodological crosswalk blocker is resolved for V1. The panel is ready for substantive descriptive analysis, with the caveat that historical and composite entities were documented and quantified rather than forced into contemporary ISO3 states.
